# 🥒 Zucchini Color Scoring — Extended ML Pipeline (v3)

**Task:** Ordinal ranking of zucchini images into 8 colour-intensity scores (1 = lightest → 8 = darkest green)  
**Dataset:** ~213 images, farmer-labelled, 8 skewed classes  
**Approach:** Multi-colour-space analysis with 3 model families × 2 pipelines each

---

## 📋 Full Pipeline

```
Raw TIF images (Score=01..08 folders)
  │
  ├─ Step 0  : Imports & Configuration           [IMG_SIZE = 128×128]
  ├─ Step 1  : Load TIF → uint8 RGB + multi-colour-space representations
  ├─ Step 2  : Inscribed-square crop
  ├─ Step 3  : Filtering comparison (None / Statistical / Z-score)
  ├─ Step 4  : Statistical analysis
  │            ├─ Per-channel histograms + KDE: BEFORE vs AFTER filtering (all 4 CS)
  │            ├─ Gaussianity tests (Shapiro-Wilk + D'Agostino)
  │            └─ Hat (Violin) diagrams: Before vs After per score class
  ├─ Step 5  : Multi-colour-space feature extraction (RGB, HSV, LAB, XYZ)
  │            ├─ 8 stats+texture features per channel (mean, std, skew, kurt,
  │            │   range, IQR, p10, p90) + 16 histogram bins = 24 features/channel
  │            ├─ 72 features per colour space, 288 combined
  │            ├─ ALL channels retained (no prior removal — see Step 11 for evidence)
  │            └─ XYZ included: Y≈L* (luminance), X/Z add chromaticity (tested Step 9.5)
  ├─ Step 6  : Visualisation — SEPARATE figure per colour space
  │            ├─ RGB: boxplots + scatter
  │            ├─ HSV: boxplots + scatter
  │            ├─ LAB: boxplots + scatter
  │            └─ XYZ: boxplots + scatter
  ├─ Step 7  : Leakage-free 80/20 train/test split
  ├─ Step 8  : Colour-safe augmentation (TRAIN only, ×16)
  ├─ Step 9  : Model-specific feature engineering pipelines
  │            ├─ SVM : LAB + HSV-S texture → SelectKBest(40) → StandardScaler
  │            ├─ RF  : HSV + LAB → variance+corr filter → NO scaling
  │            └─ XGB : All 4 CS combined → variance+corr filter → NO scaling
  ├─ Step 10 : Model training + Optuna tuning (SVM, RF, XGBoost — 2 pipelines each)
  │            ├─ 5-fold CV: train on 4 folds, validate on 5th (originals only)
  │            ├─ Overfitting diagnostic printed per fold per model
  │            ├─ XGBoost: depth≤5, high regularisation, early stopping
  │            └─ 9.5: XYZ investigation (channel correlations + ablation)
  ├─ Step 11 : Evaluation — primary metrics: R² and RMSE
  │            ├─ Accuracy NOT reported for composite-score regression (REQ 6)
  │            └─ Acc@1 shown only as secondary diagnostic
  ├─ Step 12 : Feature importance (RF + XGBoost) with channel-removal justification
  │            └─ Channels marked CANDIDATE only if importance < 0.005 in BOTH models
  ├─ Step 13 : Dimensionality reduction (PCA vs FeatSel × best 2 pipelines)
  ├─ Step 14 : Prediction frieze (per model, per score row, max 15 images)
  ├─ Step 15 : Master comparison table (Model | CS | Features | R² | RMSE | Time)
  └─ Step 16 : Final markdown analysis report
```

## ✅ v3 Changes (vs v2)

| # | Area | Change |
|---|---|---|
| REQ 1 | KNN | **Removed** — lazy learner, curse of dimensionality, no feature importance |
| REQ 2 | Pipelines | **3 models × 2 pipelines each** (A: model-specific, B: baseline for comparison) |
| REQ 2 | SVM | LAB + HSV-S texture + StandardScaler (RBF kernel needs standardised input) |
| REQ 2 | RF | HSV + LAB, no scaling (trees are scale-invariant) |
| REQ 2 | XGB | All 4 CS combined, no scaling (broadest features + strong regularisation) |
| REQ 3 | XYZ | Fully included: extraction, visualisation, stats, channel correlation + ablation |
| REQ 4 | Channel removal | **No prior removal** — evidence-based policy via Step 11 importance |
| REQ 5 | Visualisation | **Separate figure per CS** (RGB / HSV / LAB / XYZ — never combined) |
| REQ 6 | Metrics | R² and RMSE are primary; **Accuracy removed** from regression evaluation |
| REQ 7 | Accuracy | Retained only where logically appropriate (Acc@1 as diagnostic, confusion matrices) |
| REQ 8 | Final table | Model + Color space + Feature set + R² + RMSE + Training time |
| REQ 9 | Data loading | Unchanged; every modification explained inline |

> ⚠️ **Leakage rules enforced throughout:**
> - Augmentation applied after train/test split — test images never augmented
> - Feature engineering (scaler, selector, PCA) fitted on train folds only
> - CV validates on original (non-augmented) images only


## 0 · Imports & Configuration

In [ ]:
import os, glob, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import tifffile as tiff
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

from scipy.stats import (spearmanr, kendalltau, skew, kurtosis,
                          shapiro, normaltest, gaussian_kde)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (confusion_matrix, mean_squared_error,
                             mean_absolute_error, r2_score, accuracy_score)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
# KNN removed (v3): lazy learner, no feature importance, poor dimensionality scaling
from sklearn.decomposition import PCA
import xgboost as xgb

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ── Configuration ────────────────────────────────────────────────────────────
DATA_ROOT       = r'F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\nocc\CSVS01'
IMG_SIZE        = (128, 128)
K_STAT          = 1.5      # σ half-width for statistical LAB filter
FG_THRESH       = 10       # pixel value threshold to detect foreground
TEST_SPLIT      = 0.20
VAL_SPLIT       = 0.10
N_FOLDS         = 5
N_OPTUNA_TRIALS = 100
N_HIST_BINS     = 16       # configurable histogram bins for feature extraction
OUT_DIR         = 'ml_outputs'
os.makedirs(OUT_DIR, exist_ok=True)

SCORE_FOLDERS = [f'Score={str(i).zfill(2)}' for i in range(1, 9)]
IDX_TO_SCORE  = np.array([1,2,3,4,5,6,7,8], dtype=np.float32)
NUM_CLASSES   = 8

# Color spaces to analyse
COLOR_SPACES = ['RGB', 'HSV', 'LAB', 'XYZ']

def savefig(name, dpi=130):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  💾 {path}')

print('Configuration ready ✔')
print(f'  DATA_ROOT : {DATA_ROOT}')
print(f'  IMG_SIZE  : {IMG_SIZE}')
print(f'  K_STAT    : {K_STAT}')
print(f'  N_FOLDS   : {N_FOLDS}')
print(f'  N_HIST_BINS: {N_HIST_BINS}')
print(f'  COLOR_SPACES: {COLOR_SPACES}')
print(f'  Optuna trials per model: {N_OPTUNA_TRIALS}')

## 1 · Load TIF Images & Multi-Color-Space Representations

16-bit TIFF images are loaded and normalised to uint8 RGB.  
The score label is parsed directly from the folder name (`Score=01` → score 1).  
All four color-space representations are stored: RGB, HSV, LAB, XYZ.

In [ ]:
def load_tif_uint8(path: str) -> np.ndarray:
    """Load uint16 TIFF → normalise to uint8 RGB."""
    img = tiff.imread(path)
    if img.ndim == 4:
        img = img[0]
    if img.ndim == 3 and img.shape[0] in (1, 3, 4):
        img = np.transpose(img, (1, 2, 0))
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    elif img.shape[-1] == 1:
        img = np.concatenate([img]*3, axis=-1)
    elif img.shape[-1] == 4:
        img = img[..., :3]
    if img.dtype != np.uint8:
        lo, hi = img.min(), img.max()
        img = ((img.astype(np.float32) - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)
    return img


def rgb_to_lab(img_rgb: np.ndarray) -> np.ndarray:
    """uint8 RGB → uint8 LAB (OpenCV convention)."""
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)


def convert_color_space(img_rgb: np.ndarray, space: str) -> np.ndarray:
    """Convert uint8 RGB to the requested color space."""
    if space == 'RGB':
        return img_rgb.copy()
    elif space == 'HSV':
        return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    elif space == 'LAB':
        return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    elif space == 'XYZ':
        return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2XYZ)
    else:
        raise ValueError(f'Unknown color space: {space}')


def collect_images(root: str) -> tuple:
    """Load all TIF images from Score=01..08 sub-folders with multi-color-space storage."""
    rgb_list, score_list, path_list = [], [], []
    cs_images = {cs: [] for cs in COLOR_SPACES}
    for folder in SCORE_FOLDERS:
        score_int = int(folder.split('=')[1])
        patterns = [os.path.join(root, '**', folder, ext)
                    for ext in ['*.tif','*.TIF','*.tiff','*.TIFF']]
        paths = []
        for pat in patterns:
            paths += glob.glob(pat, recursive=True)
        paths = list(set(paths))
        for p in paths:
            try:
                rgb = load_tif_uint8(p)
                rgb = cv2.resize(rgb, (IMG_SIZE[1], IMG_SIZE[0]), interpolation=cv2.INTER_AREA)
                rgb_list.append(rgb)
                for cs in COLOR_SPACES:
                    cs_images[cs].append(convert_color_space(rgb, cs))
                score_list.append(score_int); path_list.append(p)
            except Exception as e:
                print(f'  ⚠ skip {p}: {e}')
    X_rgb = np.array(rgb_list, dtype=np.uint8)
    X_cs = {cs: np.array(cs_images[cs], dtype=np.uint8) for cs in COLOR_SPACES}
    y     = np.array(score_list, dtype=np.int32)
    df    = pd.DataFrame({'path': path_list, 'score': score_list})
    print(f'Loaded {len(df)} images')
    print(df.groupby('score')['path'].count().rename('count').to_string())
    return X_rgb, X_cs, y, df


X_rgb, X_cs_all, y_raw, df_all = collect_images(DATA_ROOT)
X_lab = X_cs_all['LAB']

In [ ]:
# ── Class distribution plot ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
counts = np.bincount(y_raw, minlength=9)[1:]
colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, NUM_CLASSES))
bars = ax.bar([f'Score {i+1}' for i in range(NUM_CLASSES)], counts, color=colors, edgecolor='white')
for b, c in zip(bars, counts):
    ax.text(b.get_x() + b.get_width()/2, c + 0.3, str(c), ha='center', fontsize=10, fontweight='bold')
ax.set_title('Dataset Class Distribution (213 images, farmer-labeled)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of images'); ax.set_xlabel('Score')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); savefig('00_class_distribution.png'); plt.show()


## 2 · Inscribed-Square Crop

Each image is cropped to the largest square inscribed in the largest inscribed circle of the foreground region.  
This removes background noise and image borders while keeping only the central zucchini area.  

**Why inscribed-square crop?**  
- Eliminates non-zucchini pixels (borders, shadows, background)
- Produces a consistent square input regardless of original image shape
- Pixels outside the inscribed circle are set to white (neutral, not black, to avoid biasing LAB stats)


In [ ]:
def get_inscribed_square_crop(img: np.ndarray) -> tuple:
    """
    Inscribed-square crop:
    1. Detect foreground pixels (any channel > FG_THRESH).
    2. Distance transform → largest inscribed circle centre.
    3. Crop the largest square inscribed in that circle; corners → white.
    Returns: (cropped_img, info_dict)
    """
    H, W = img.shape[:2]
    fg_mask = np.any(img > FG_THRESH, axis=-1).astype(np.uint8)
    dist = cv2.distanceTransform(fg_mask, cv2.DIST_L2, 5)
    _, r_f, _, (cx, cy) = cv2.minMaxLoc(dist)
    r = int(r_f); cx, cy = int(cx), int(cy)
    side = int(r * 2**0.5); side = side - (side % 2)
    sx1 = max(0, cx - side//2); sy1 = max(0, cy - side//2)
    sx2 = min(W, sx1 + side);   sy2 = min(H, sy1 + side)
    side = min(sx2-sx1, sy2-sy1); sx2, sy2 = sx1+side, sy1+side
    patch = img[sy1:sy2, sx1:sx2].copy()
    gy, gx = np.mgrid[sy1:sy2, sx1:sx2]
    outside = ((gx - cx)**2 + (gy - cy)**2) > r**2
    patch[outside] = 255
    return patch, {'cx':cx,'cy':cy,'r':r,'side':side,'sx1':sx1,'sy1':sy1,'sx2':sx2,'sy2':sy2}


# Apply to all images
X_rgb_cropped = []
crop_infos    = []
for img in tqdm(X_rgb, desc='Cropping'):
    c, info = get_inscribed_square_crop(img)
    X_rgb_cropped.append(cv2.resize(c, (IMG_SIZE[1], IMG_SIZE[0]), interpolation=cv2.INTER_AREA))
    crop_infos.append(info)
X_rgb_cropped = np.array(X_rgb_cropped, dtype=np.uint8)

# Build all color-space representations of cropped images
X_cs_cropped = {}
for cs in COLOR_SPACES:
    X_cs_cropped[cs] = np.array([convert_color_space(r, cs) for r in X_rgb_cropped], dtype=np.uint8)
X_lab_cropped = X_cs_cropped['LAB']
print(f'Cropped: {X_rgb_cropped.shape}')

In [ ]:
# ── Crop visualisation: one sample per score ─────────────────────────────────
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(20, 5))
for i in range(NUM_CLASSES):
    idx = np.where(y_raw == i+1)[0][0]
    axes[0, i].imshow(X_rgb[idx])
    axes[0, i].set_title(f'Score {i+1}\nOriginal', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(X_rgb_cropped[idx])
    axes[1, i].set_title('Cropped', fontsize=8)
    axes[1, i].axis('off')
plt.suptitle('Inscribed-Square Crop — One Sample per Score', fontsize=12, fontweight='bold')
plt.tight_layout(); savefig('01_crop_visualisation.png'); plt.show()


## 3 · Filtering Comparison

Three filtering strategies are compared:
1. **No filtering** — raw foreground pixels
2. **Statistical LAB filter** (K=1.5σ) — existing pipeline filter
3. **Z-score filter** (|z| < 2.0) — per-channel outlier removal

For each image, foreground pixels are kept only if their L*, a*, b* values fall within the filter bounds.
Rejected pixels are set to white.

In [ ]:
def stat_filter(img_rgb: np.ndarray, k: float = K_STAT) -> dict:
    """Statistical LAB interval filter. Keeps pixels where each LAB channel is within [mean ± k·σ]."""
    H, W = img_rgb.shape[:2]
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[...,0] * 100.0 / 255.0
    a_img = lab[...,1] - 128.0
    b_img = lab[...,2] - 128.0
    coords = np.argwhere(fg_mask)
    if len(coords) == 0:
        return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0}
    r, c = coords[:,0], coords[:,1]
    keep_px = np.ones(len(coords), dtype=bool)
    for ch_img in [L_img, a_img, b_img]:
        vals = ch_img[r, c]
        med, std = vals.mean(), vals.std()
        keep_px &= (vals >= med - k*std) & (vals <= med + k*std)
    kept_mask = np.zeros((H, W), bool)
    kept_mask[r[keep_px], c[keep_px]] = True
    out = np.full_like(img_rgb, 255)
    out[kept_mask] = img_rgb[kept_mask]
    return {'out': out, 'kept_mask': kept_mask,
            'kept_pct': float(keep_px.sum() / len(coords) * 100)}


def zscore_filter(img_rgb: np.ndarray, z_thresh: float = 2.0) -> dict:
    """Z-score filter: keeps pixels where |z| < z_thresh for all LAB channels."""
    H, W = img_rgb.shape[:2]
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_img = lab[...,0] * 100.0 / 255.0
    a_img = lab[...,1] - 128.0
    b_img = lab[...,2] - 128.0
    coords = np.argwhere(fg_mask)
    if len(coords) == 0:
        return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0}
    r, c = coords[:,0], coords[:,1]
    keep_px = np.ones(len(coords), dtype=bool)
    for ch_img in [L_img, a_img, b_img]:
        vals = ch_img[r, c]
        mu, sigma = vals.mean(), vals.std()
        if sigma > 1e-6:
            z = np.abs((vals - mu) / sigma)
            keep_px &= (z < z_thresh)
    kept_mask = np.zeros((H, W), bool)
    kept_mask[r[keep_px], c[keep_px]] = True
    out = np.full_like(img_rgb, 255)
    out[kept_mask] = img_rgb[kept_mask]
    return {'out': out, 'kept_mask': kept_mask,
            'kept_pct': float(keep_px.sum() / len(coords) * 100)}


def no_filter(img_rgb: np.ndarray) -> dict:
    """No filtering — keep all foreground pixels."""
    fg_mask = np.any(img_rgb > FG_THRESH, axis=-1)
    return {'out': img_rgb.copy(), 'kept_mask': fg_mask, 'kept_pct': 100.0}


# Apply all three filtering strategies
FILTER_METHODS = {
    'No Filter': no_filter,
    'Statistical (K=1.5σ)': lambda img: stat_filter(img, k=K_STAT),
    'Z-score (|z|<2)': lambda img: zscore_filter(img, z_thresh=2.0),
}

filter_outputs = {}
for fname, ffunc in FILTER_METHODS.items():
    results_f = [ffunc(img) for img in tqdm(X_rgb_cropped, desc=f'Filter: {fname}')]
    filter_outputs[fname] = {
        'images': np.array([r['out'] for r in results_f], dtype=np.uint8),
        'masks': np.array([r['kept_mask'] for r in results_f], dtype=bool),
        'kept_pcts': np.array([r['kept_pct'] for r in results_f], dtype=np.float32),
    }
    pcts = filter_outputs[fname]['kept_pcts']
    print(f'  {fname}: kept mean={pcts.mean():.1f}%  std={pcts.std():.1f}%')

# Primary filtered set (statistical filter, as in original pipeline)
X_rgb_filtered = filter_outputs['Statistical (K=1.5σ)']['images']
X_masks = filter_outputs['Statistical (K=1.5σ)']['masks']
kept_pcts = filter_outputs['Statistical (K=1.5σ)']['kept_pcts']

In [ ]:
# ── Filtering comparison visualisation ────────────────────────────────────────
fig, axes = plt.subplots(len(FILTER_METHODS), NUM_CLASSES, figsize=(20, 3*len(FILTER_METHODS)))
for row, (fname, fdata) in enumerate(filter_outputs.items()):
    for col in range(NUM_CLASSES):
        idx = np.where(y_raw == col+1)[0][0]
        axes[row, col].imshow(fdata['images'][idx])
        pct = fdata['kept_pcts'][idx]
        axes[row, col].set_title(f'S{col+1} ({pct:.0f}%)', fontsize=8)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(fname, fontsize=9, fontweight='bold')
plt.suptitle('Filtering Comparison — One Sample per Score', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig('02_filter_comparison.png'); plt.show()

# ── Distribution comparison before/after filtering ──────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
ch_names = ['L*', 'a*', 'b*']
for col, ch_name in enumerate(ch_names):
    for row, (fname, fdata) in enumerate(filter_outputs.items()):
        all_vals = []
        for i in range(len(X_rgb_cropped)):
            lab = cv2.cvtColor(fdata['images'][i], cv2.COLOR_RGB2LAB).astype(np.float32)
            ch_vals = [lab[...,0]*100.0/255.0, lab[...,1]-128.0, lab[...,2]-128.0][col]
            fg = fdata['masks'][i]
            all_vals.extend(ch_vals[fg].ravel()[:500])
        axes[row, col].hist(all_vals, bins=60, density=True, alpha=0.7, color='steelblue', edgecolor='white')
        axes[row, col].set_title(f'{fname} — {ch_name}', fontsize=9, fontweight='bold')
        axes[row, col].set_xlabel(ch_name); axes[row, col].grid(alpha=0.3)
plt.suptitle('Pixel Distribution by Filtering Method (LAB channels)', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig('02b_filter_distributions.png'); plt.show()

## 4 · Statistical Analysis

**New in v2:** Full before/after comparison across all 4 color spaces to help choose the best one.

### 4a · Per-channel Histograms + KDE: Before vs After Filtering (all color spaces)
For **each color space** (RGB, HSV, LAB, XYZ):
- **Top row (red):** raw cropped foreground pixel distribution
- **Bottom row (blue):** after statistical LAB filter (K=1.5σ)
- μ, σ, skewness, kurtosis annotated per subplot

**How to use these panels to choose a color space:**
- Prefer channels where filtering **narrows** the distribution (↓ σ)
- Prefer channels where distributions **monotonically shift** across scores (discriminative)
- Prefer channels with **lower skewness** after filtering (closer to Gaussian → benefits linear models)

### 4b · Gaussianity Tests: Shapiro-Wilk + D'Agostino Before vs After
- **Shapiro-Wilk:** checks normality (p > 0.05 → Gaussian)
- **D'Agostino-Pearson:** tests skewness + kurtosis jointly
- Before vs After comparison shows how filtering affects distribution shape

### 4c · Hat (Violin) Diagrams: Before vs After, Per Score Class
- **Red violins:** before filtering
- **Blue violins:** after filtering
- One figure per color space; shows whether filtering tightens intra-class variance


In [ ]:
# ── Per-channel histograms + KDE: BEFORE vs AFTER filtering — ALL color spaces ──
# For each color space: 2 rows (Before / After) × 3 channels
# Helps choose the best color space: look for tightest / most Gaussian distributions after filtering

CS_CHANNEL_NAMES = {
    'RGB': ['R', 'G', 'B'],
    'HSV': ['H', 'S', 'V'],
    'LAB': ['L*', 'a*', 'b*'],
    'XYZ': ['X', 'Y', 'Z'],
}

def get_physical_channels(img_rgb, fg_mask, space):
    """Extract physical-scale channel values from foreground pixels."""
    cs_img = convert_color_space(img_rgb, space).astype(np.float32)
    if space == 'LAB':
        cs_img[...,0] = cs_img[...,0] * 100.0 / 255.0
        cs_img[...,1] = cs_img[...,1] - 128.0
        cs_img[...,2] = cs_img[...,2] - 128.0
    channels = []
    for c in range(3):
        channels.append(cs_img[..., c][fg_mask])
    return channels


raw_masks = filter_outputs['No Filter']['masks']

for cs in COLOR_SPACES:
    ch_names = CS_CHANNEL_NAMES[cs]
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    fig.suptitle(
        f'{cs} — Per-Channel Histograms + KDE: Before vs After Statistical Filtering\n'
        f'(Red = Before  |  Blue = After  |  Black line = KDE)\n'
        f'Use this panel to assess distribution shape and tightening effect of filtering',
        fontsize=13, fontweight='bold')

    for col in range(3):
        for row, (label, imgs, masks, color, linestyle) in enumerate([
            ('Before filtering', X_rgb_cropped, raw_masks,      '#e74c3c', '-'),
            ('After filtering',  X_rgb_filtered, X_masks,       '#2980b9', '--'),
        ]):
            all_vals = []
            for i in range(min(80, len(imgs))):
                chs = get_physical_channels(imgs[i], masks[i], cs)
                all_vals.extend(chs[col][:400].tolist())
            vals = np.array(all_vals)
            ax = axes[row, col]
            ax.hist(vals, bins=60, density=True, alpha=0.50,
                    color=color, edgecolor='white', label='Histogram')
            if len(vals) > 10 and vals.std() > 1e-6:
                kde = gaussian_kde(vals)
                x_range = np.linspace(vals.min(), vals.max(), 300)
                ax.plot(x_range, kde(x_range), 'k-', lw=2, label='KDE')
                # Mark mean ± std
                mu, sd = vals.mean(), vals.std()
                ax.axvline(mu, color=color, lw=1.5, linestyle='--', alpha=0.9, label=f'μ={mu:.1f}')
                ax.axvline(mu - sd, color=color, lw=1, linestyle=':', alpha=0.7)
                ax.axvline(mu + sd, color=color, lw=1, linestyle=':', alpha=0.7)
                sk_val = float(skew(vals))
                ku_val = float(kurtosis(vals))
                ax.text(0.97, 0.95, f'μ={mu:.2f}  σ={sd:.2f}\nskew={sk_val:.2f}  kurt={ku_val:.2f}',
                        transform=ax.transAxes, ha='right', va='top', fontsize=7,
                        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
            ax.set_title(f'{label} — {ch_names[col]}', fontsize=10, fontweight='bold')
            ax.legend(fontsize=7); ax.grid(alpha=0.3)
            ax.set_xlabel(ch_names[col])

    plt.tight_layout()
    savefig(f'03a_{cs}_before_after_kde.png')
    plt.show()

print('\n📌 Color-space selection guide:')
print('  → Look for AFTER-filtering distributions that are:')
print('     • Narrow (small σ) — less intra-class variance')
print('     • Low absolute skewness — closer to Gaussian')
print('     • Monotonically shifting across scores (discriminative)')


In [ ]:
# ── Gaussianity tests: BEFORE vs AFTER filtering, all color spaces ───────────
gaussianity_rows = []
for stage, imgs, masks in [
    ('Before Filter', X_rgb_cropped, raw_masks),
    ('After Filter',  X_rgb_filtered, X_masks),
]:
    for cs in COLOR_SPACES:
        ch_names = CS_CHANNEL_NAMES[cs]
        for ch_idx, ch_name in enumerate(ch_names):
            all_vals = []
            for i in range(len(imgs)):
                chs = get_physical_channels(imgs[i], masks[i], cs)
                all_vals.extend(chs[ch_idx][:200].tolist())
            vals = np.array(all_vals)
            mu = vals.mean(); sd = vals.std()
            sk = float(skew(vals)); ku = float(kurtosis(vals))
            sub = vals[np.random.choice(len(vals), min(5000, len(vals)), replace=False)]
            sw_stat, sw_p = shapiro(sub)
            if len(sub) >= 20:
                da_stat, da_p = normaltest(sub)
            else:
                da_stat, da_p = np.nan, np.nan
            gaussianity_rows.append({
                'Stage': stage, 'Color Space': cs, 'Channel': ch_name,
                'Mean': round(mu, 3), 'Std': round(sd, 3),
                'Skewness': round(sk, 3), 'Kurtosis': round(ku, 3),
                'Shapiro W': round(sw_stat, 4), 'Shapiro p': f'{sw_p:.2e}',
                'DAgostino K²': round(da_stat, 2) if not np.isnan(da_stat) else 'N/A',
                'DAgostino p': f'{da_p:.2e}' if not np.isnan(da_p) else 'N/A',
                'Gaussian?': 'Yes' if sw_p > 0.05 else 'No',
            })

df_gauss = pd.DataFrame(gaussianity_rows)
print('\n📊 Gaussianity Analysis — Before vs After Filtering\n')
print(df_gauss.to_string(index=False))
df_gauss.to_csv(os.path.join(OUT_DIR, 'gaussianity_tests.csv'), index=False)

# Summary: which color space has most Gaussian channels after filtering?
after_df = df_gauss[df_gauss['Stage'] == 'After Filter']
gauss_by_cs = after_df.groupby('Color Space')['Gaussian?'].apply(lambda x: (x == 'Yes').sum())
print('\n📊 Gaussian channels after filtering (count out of 3):')
for cs_name, cnt in gauss_by_cs.sort_values(ascending=False).items():
    print(f'  {cs_name}: {cnt}/3 channels Gaussian')

# Before vs After comparison
print('\n📊 Filtering impact on Gaussianity:')
for cs_name in COLOR_SPACES:
    before = df_gauss[(df_gauss['Stage'] == 'Before Filter') & (df_gauss['Color Space'] == cs_name)]
    after  = df_gauss[(df_gauss['Stage'] == 'After Filter')  & (df_gauss['Color Space'] == cs_name)]
    b_cnt = (before['Gaussian?'] == 'Yes').sum()
    a_cnt = (after['Gaussian?'] == 'Yes').sum()
    delta = a_cnt - b_cnt
    sign = '+' if delta > 0 else ''
    print(f'  {cs_name}: {b_cnt}/3 → {a_cnt}/3 ({sign}{delta})')

In [ ]:
# ── Hat (Violin) Diagrams — per-score channel distributions, Before vs After ──
# Each subplot: two sets of violins (before=red, after=blue) side by side
# Helps visualise how filtering tightens the distribution per score class

for cs in COLOR_SPACES:
    ch_names = CS_CHANNEL_NAMES[cs]
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle(
        f'{cs} — Hat (Violin) Diagrams by Score Class: Before vs After Filtering\n'
        f'(Red = Before filtering  |  Blue = After filtering)',
        fontsize=13, fontweight='bold')

    for col, ch_name in enumerate(ch_names):
        ax = axes[col]
        # Collect per-score per-stage channel means
        for stage_i, (stage_label, imgs, masks, base_color) in enumerate([
            ('Before', X_rgb_cropped,  raw_masks, '#e74c3c'),
            ('After',  X_rgb_filtered, X_masks,   '#2980b9'),
        ]):
            score_vals = {s: [] for s in sorted(set(y_raw))}
            for i in range(len(imgs)):
                chs = get_physical_channels(imgs[i], masks[i], cs)
                ch_mean = float(np.mean(chs[col])) if len(chs[col]) > 0 else 0.0
                score_vals[y_raw[i]].append(ch_mean)

            data_list = [score_vals[s] for s in sorted(score_vals.keys())]
            n_scores  = len(data_list)
            # Offset positions so before and after sit side-by-side
            offset = -0.22 + stage_i * 0.44
            positions = [j + offset for j in range(n_scores)]

            parts = ax.violinplot(
                data_list, positions=positions, widths=0.38,
                showmeans=True, showmedians=True, showextrema=True)
            for pc in parts['bodies']:
                pc.set_facecolor(base_color)
                pc.set_alpha(0.55)
                pc.set_edgecolor(base_color)
            for key in ['cmeans', 'cmedians', 'cbars', 'cmaxes', 'cmins']:
                if key in parts:
                    parts[key].set_color(base_color)
                    parts[key].set_linewidth(1.2)

        ax.set_xticks(range(n_scores))
        ax.set_xticklabels([str(s) for s in sorted(set(y_raw))], fontsize=9)
        ax.set_xlabel('Score', fontsize=10)
        ax.set_ylabel(f'{ch_name} mean value', fontsize=10)
        ax.set_title(f'{ch_name}', fontsize=11, fontweight='bold')
        ax.grid(alpha=0.3, axis='y')

        # Legend patches
        import matplotlib.patches as mpatches
        ax.legend(handles=[
            mpatches.Patch(color='#e74c3c', alpha=0.6, label='Before filtering'),
            mpatches.Patch(color='#2980b9', alpha=0.6, label='After filtering'),
        ], fontsize=8, loc='upper right')

    plt.tight_layout()
    savefig(f'03b_{cs}_violin_hat.png')
    plt.show()


## 5 · Multi-Color-Space Feature Extraction

### Color spaces used and why

| CS | Channels | Rationale |
|---|---|---|
| **RGB** | R, G, B | Baseline; human-interpretable color; reference for XYZ conversion |
| **HSV** | H (hue), S (saturation), V (value/brightness) | Separates color identity from illumination; H tracks greenness category |
| **LAB** | L* (lightness), a* (red–green), b* (yellow–blue) | Perceptually uniform; a* directly encodes the green–red axis relevant to zucchini ripeness |
| **XYZ** | X (luminance-weighted R+G), Y (pure luminance = L), Z (blue-weighted) | CIE standard tristimulus; Y is an alternative luminance measure independent of sRGB primaries; X and Z add chromaticity dimensions not directly present in RGB/LAB |

> **Why include XYZ?** XYZ is the parent space from which LAB is derived. Y is mathematically identical to perceived luminance and often correlates with ripeness. X and Z provide orthogonal chromaticity information. Their contribution relative to RGB/HSV/LAB is empirically tested in Step 14.1 (color-space ablation) and Step 11 (feature importance). If XYZ features do not rank in the top-K, this is documented there.

### Feature engineering per channel

Per channel, we extract **8 statistical/texture features + N_HIST_BINS histogram bins**:

| Group | Features | Purpose |
|---|---|---|
| Moments | mean, std, skewness, kurtosis | Central tendency and distribution shape |
| Texture proxies | range, IQR, p10, p90 | Distribution spread; captures spottiness and non-uniformity of zucchini surface |
| Histogram | 16 bin frequencies (normalized) | Full distribution shape; discriminates multimodal surfaces |

> **Channel removal policy:** No channel is pre-removed. The previous approach of removing LAB-b or HSV-V was not justified by evidence — they are retained here. See Step 11 (feature importance) and Step 14 (channel ablation) for post-hoc evidence. Only channels with near-zero variance *across the entire dataset* would be candidates for removal.

**Total features:** 3 channels × 24 per channel = **72 per color space**, **288 combined**.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Feature extraction helpers
# Per-channel: mean, std, skewness, kurtosis + n_bins histogram bins
# Texture proxy: range (max-min), IQR (inter-quartile range), p10, p90
# These are added for every channel to enrich the feature space.
# Rationale: zucchini color "texture" (spottiness, gradients) affects perceived
# quality and is captured by distribution spread features even without GLCM.
# ─────────────────────────────────────────────────────────────────────────────

def extract_features_single_space(img_rgb: np.ndarray, fg_mask: np.ndarray,
                                   space: str, n_bins: int = N_HIST_BINS) -> np.ndarray:
    """
    Extract features for one color space from filtered foreground pixels.
    Per channel (physical scale):
      - 4 statistics : mean, std, skewness, kurtosis
      - 4 texture    : range (max-min), IQR (p75-p25), p10, p90
      - n_bins hist  : normalized histogram frequencies
    Returns array of shape (3 * (8 + n_bins),).
    """
    channels = get_physical_channels(img_rgb, fg_mask, space)
    feats = []
    for vals in channels:
        if len(vals) == 0:
            feats.extend([0.0] * (8 + n_bins))
            continue
        # Statistical moments
        feats.append(float(vals.mean()))
        feats.append(float(vals.std()))
        feats.append(float(skew(vals)))
        feats.append(float(kurtosis(vals)))
        # Texture / distribution shape proxies
        p10, p25, p75, p90 = np.percentile(vals, [10, 25, 75, 90])
        feats.append(float(float(vals.max()) - float(vals.min())))  # range
        feats.append(float(p75 - p25))                               # IQR
        feats.append(float(p10))
        feats.append(float(p90))
        # Histogram bin features (normalized frequencies)
        hist, _ = np.histogram(vals, bins=n_bins, density=True)
        feats.extend(hist.tolist())
    return np.array(feats, dtype=np.float32)


def extract_all_features(img_rgb: np.ndarray, fg_mask: np.ndarray,
                         n_bins: int = N_HIST_BINS) -> dict:
    """Extract features for all color spaces. Returns dict of arrays + combined."""
    cs_feats = {}
    for cs in COLOR_SPACES:
        cs_feats[cs] = extract_features_single_space(img_rgb, fg_mask, cs, n_bins)
    cs_feats['Combined'] = np.concatenate([cs_feats[cs] for cs in COLOR_SPACES])
    return cs_feats


def build_feature_names(space: str, n_bins: int = N_HIST_BINS) -> list:
    """Build feature names for one color space (8 stats + n_bins histogram)."""
    ch_names = CS_CHANNEL_NAMES[space]
    names = []
    for ch in ch_names:
        names += [f'{space}_{ch}_mean', f'{space}_{ch}_std',
                  f'{space}_{ch}_skew', f'{space}_{ch}_kurt',
                  f'{space}_{ch}_range', f'{space}_{ch}_iqr',
                  f'{space}_{ch}_p10',  f'{space}_{ch}_p90']
        names += [f'{space}_{ch}_bin{b}' for b in range(n_bins)]
    return names


# Derived constant: features per channel changed from 4+n_bins to 8+n_bins
N_FEATS_PER_CH = 8 + N_HIST_BINS   # 24 per channel, 72 per CS, 288 combined

# Build all feature name lists
feat_names_per_cs = {cs: build_feature_names(cs) for cs in COLOR_SPACES}
feat_names_combined = []
for cs in COLOR_SPACES:
    feat_names_combined.extend(feat_names_per_cs[cs])

print(f'Feature dimensions per color space  : {N_FEATS_PER_CH * 3}')
print(f'Feature dimensions combined (4 CS)  : {N_FEATS_PER_CH * 3 * 4}')
print(f'Feature names built                 : {len(feat_names_combined)}')
print()
print('  ⚙  Texture proxy features added: range, IQR, p10, p90 per channel')
print('  ⚙  XYZ fully included from CS_CHANNEL_NAMES + extract_all_features')
print('  ⚙  No channels pre-removed — see Step 11 for importance-based justification')


In [ ]:
# ── 5 · Run feature extraction over ALL filtered images ─────────────────────
# Builds X_feats_per_cs, X_feats_combined, X_feats (LAB-only, backward-compat)
# for every image in the dataset (using the stat-filtered images + masks).
#
# IMPORTANT: This must run before the boxplot / scatter visualisation cells
# (cells 5a and 5b) and before the train/test split cell.
# No leakage here: we are only extracting features, not fitting any model.

X_feats_per_cs  = {cs: [] for cs in COLOR_SPACES}  # per-CS feature arrays
X_feats_combined_list = []
X_feats_list    = []   # LAB-only (backward-compatible name)
X_feats         = None  # filled below

for i in tqdm(range(len(X_rgb_filtered)), desc='Extracting features'):
    img_rgb = X_rgb_filtered[i]
    mask    = X_masks[i]
    all_f   = extract_all_features(img_rgb, mask)
    for cs in COLOR_SPACES:
        X_feats_per_cs[cs].append(all_f[cs])
    X_feats_combined_list.append(all_f['Combined'])
    X_feats_list.append(all_f['LAB'])   # LAB-only slice

# Convert to numpy arrays
for cs in COLOR_SPACES:
    X_feats_per_cs[cs] = np.array(X_feats_per_cs[cs], dtype=np.float32)
X_feats_combined = np.array(X_feats_combined_list, dtype=np.float32)
X_feats          = np.array(X_feats_list,          dtype=np.float32)

print(f'Feature extraction complete:')
for cs in COLOR_SPACES:
    print(f'  {cs:>4}: {X_feats_per_cs[cs].shape}')
print(f'  Combined : {X_feats_combined.shape}')
print(f'  LAB only : {X_feats.shape}')


In [ ]:
# ── 5a · Per-class boxplots — ONE FIGURE PER COLOR SPACE ─────────────────────
# REQ 5: Do not combine color spaces into a single visualization figure.
# Each CS gets its own 1×3 grid (one subplot per channel).
# Index offset for "mean" feature = 0 (first feature per channel block).
# N_FEATS_PER_CH = 8 + N_HIST_BINS features per channel.
FEATS_PER_CH = 8 + N_HIST_BINS
colors_score = plt.cm.RdYlGn(np.linspace(0.1, 0.9, NUM_CLASSES))

for cs in COLOR_SPACES:
    ch_names = CS_CHANNEL_NAMES[cs]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f'{cs} — Channel Mean Distribution per Score Class\n'
        f'(boxes coloured Score 1=lightest → 8=darkest green)',
        fontsize=13, fontweight='bold')

    for col_idx, ch in enumerate(ch_names):
        ax = axes[col_idx]
        feat_idx = col_idx * FEATS_PER_CH   # index of "mean" for this channel
        per_class = [X_feats_per_cs[cs][y_raw == s+1, feat_idx] for s in range(NUM_CLASSES)]
        bp = ax.boxplot(per_class, patch_artist=True, notch=False,
                        medianprops=dict(color='black', linewidth=2))
        for patch, col in zip(bp['boxes'], colors_score):
            patch.set_facecolor(col); patch.set_alpha(0.75)
        ax.set_xticklabels([f'S{i+1}' for i in range(NUM_CLASSES)], fontsize=9)
        ax.set_ylabel(f'{cs} {ch} mean (physical scale)', fontsize=10)
        ax.set_title(f'{cs} — {ch} channel', fontsize=11, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    # Output files: 04_boxplot_RGB.png, 04_boxplot_HSV.png, 04_boxplot_LAB.png, 04_boxplot_XYZ.png
    savefig(f'04_boxplot_{cs}.png')
    plt.show()
    print(f'  ✔ {cs} boxplot saved')


In [ ]:
# XYZ — Channel-Pair Scatter is produced in the loop below for cs='XYZ'
# ── 5b · Scatter plots — ONE FIGURE PER COLOR SPACE ──────────────────────────
# REQ 5: Each color space on a separate row/section.
# Shows channel-pair scatter colored by score, plus XYZ explicitly visualized.
FEATS_PER_CH = 8 + N_HIST_BINS
cmap_scores = plt.cm.RdYlGn

cs_scatter_pairs = {
    'RGB': [('R', 'G', 0, 1), ('G', 'B', 1, 2), ('R', 'B', 0, 2)],
    'HSV': [('H', 'S', 0, 1), ('H', 'V', 0, 2), ('S', 'V', 1, 2)],
    'LAB': [('L*', 'a*', 0, 1), ('L*', 'b*', 0, 2), ('a*', 'b*', 1, 2)],
    'XYZ': [('X', 'Y', 0, 1), ('X', 'Z', 0, 2), ('Y', 'Z', 1, 2)],
}

for cs in COLOR_SPACES:
    pairs = cs_scatter_pairs[cs]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f'{cs} — Channel-Pair Scatter Coloured by Score\n'
        f'(Score 1=lightest green → 8=darkest)',
        fontsize=13, fontweight='bold')

    for ax, (ch_a, ch_b, ci_a, ci_b) in zip(axes, pairs):
        feat_a = X_feats_per_cs[cs][:, ci_a * FEATS_PER_CH]   # mean
        feat_b = X_feats_per_cs[cs][:, ci_b * FEATS_PER_CH]   # mean
        sc = ax.scatter(feat_a, feat_b, c=y_raw, cmap=cmap_scores,
                        vmin=1, vmax=8, alpha=0.85,
                        edgecolors='white', linewidths=0.3, s=60)
        ax.set_xlabel(f'{cs} {ch_a} mean', fontsize=10)
        ax.set_ylabel(f'{cs} {ch_b} mean', fontsize=10)
        ax.set_title(f'{ch_a} vs {ch_b}', fontsize=11, fontweight='bold')
        plt.colorbar(sc, ax=ax, label='Score')
        ax.grid(alpha=0.2)

    plt.tight_layout()
    # Output files: 05_scatter_RGB.png, 05_scatter_HSV.png, 05_scatter_LAB.png, 05_scatter_XYZ.png
    savefig(f'05_scatter_{cs}.png')
    plt.show()
    print(f'  ✔ {cs} scatter saved')

print('\n📌 XYZ interpretation guide:')
print('  Y channel  ≈ CIE luminance (same info as LAB L* but different scale)')
print('  X channel  ≈ weighted sum biased toward red; XZ jointly encode chromaticity')
print('  If XYZ scatter does not show score separation → low discriminative power vs LAB')


## 6 · Leakage-Free Train / Test Split

> ⚠️ **Critical leakage rule:** The split is performed on **original images** before any augmentation.
> Test images are **never augmented**, **never used** to fit any transformer or selector.

Stratified **80/20** split ensures proportional representation across all score classes:
- **Train (80%):** augmented ×16, used for CV training and hyperparameter tuning
- **Test (20%):** held-out final evaluation — never seen during training or tuning

**5-Fold Stratified CV** on the training set is used for all model selection decisions.
CV folds are split by **original image identity** — all augmented variants of one image stay together
in the same fold. Validation folds use **only original (non-augmented)** images to prevent leakage.


In [ ]:
idx_all = np.arange(len(X_feats_combined))

# 80% train / 20% test — stratified
idx_tr, idx_te = train_test_split(
    idx_all, test_size=0.20, stratify=y_raw, random_state=SEED)

# Per-color-space splits
X_feats_tr_per_cs = {cs: X_feats_per_cs[cs][idx_tr] for cs in COLOR_SPACES}
X_feats_te_per_cs = {cs: X_feats_per_cs[cs][idx_te] for cs in COLOR_SPACES}
X_feats_tr_combined_orig = X_feats_combined[idx_tr]
X_feats_te_combined = X_feats_combined[idx_te]

# Backward-compatible LAB-only
X_feats_tr_orig = X_feats[idx_tr]; y_tr_orig = y_raw[idx_tr]
X_rgb_tr_orig   = X_rgb_filtered[idx_tr]
X_lab_tr_orig   = np.array([rgb_to_lab(r) for r in X_rgb_tr_orig])
X_feats_te      = X_feats[idx_te]; y_te = y_raw[idx_te]

# Also build per-filter-method features for train/test
X_feats_per_filter = {}
for fname, fdata in filter_outputs.items():
    f_feats = []
    for i in range(len(X_rgb_cropped)):
        f_feats.append(extract_features_single_space(
            fdata['images'][i], fdata['masks'][i], 'LAB'))
    f_feats = np.array(f_feats, dtype=np.float32)
    X_feats_per_filter[fname] = {
        'train': f_feats[idx_tr],
        'test': f_feats[idx_te],
    }

print(f'Train (original): {len(idx_tr)} images  ({len(idx_tr)/len(idx_all)*100:.0f}%)')
print(f'Test            : {len(idx_te)} images  ({len(idx_te)/len(idx_all)*100:.0f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lbl, color, y_split in [
    (axes[0], 'Train',  'steelblue', y_tr_orig),
    (axes[1], 'Test',   'seagreen',  y_te)]:
    cnt = np.bincount(y_split, minlength=9)[1:]
    bars = ax.bar([f'S{i+1}' for i in range(NUM_CLASSES)], cnt, color=color, edgecolor='white')
    for b, c in zip(bars, cnt):
        ax.text(b.get_x()+b.get_width()/2, c+0.2, str(c), ha='center', fontsize=9)
    ax.set_title(f'{lbl} ({len(y_split)} images)', fontweight='bold')
    ax.set_ylim(0, cnt.max()+5); ax.grid(axis='y', alpha=0.3)
plt.suptitle('Stratified 80/20 Split (Before Augmentation)', fontsize=12, fontweight='bold')
plt.tight_layout(); savefig('06_split_distribution.png'); plt.show()

## 7 · Colour-Safe Augmentation (Train Only)

Augmentation is applied **only to training images** to prevent data leakage.  

**Allowed augmentations** (colour-identity preserving):

| Type | Variants | Applied on | Rationale |
|---|---|---|---|
| Flip | H / V / HV | RGB | Does not change colour |
| Rotation | +15° / −15° / +30° | RGB | Preserves LAB values |
| Crop/Zoom | +10% / +20% / +30% | RGB | Spatial only |
| Blur | σ=0.5 / 1.0 / 1.5 | LAB | Smooths texture, keeps colour |
| Noise | σ=3 / 6 / 9 | LAB | Adds robustness |

**Forbidden augmentations:** `HueSaturationValue`, `ColorJitter` with hue/sat, `RandomGamma`  
— these directly perturb the colour intensity being measured, creating false labels.

After each augmentation: stat_filter is re-applied and features re-extracted from filtered pixels.  
This multiplies the training set by **×16** (1 original + 15 variants).


In [ ]:
# ── Augmentation functions ───────────────────────────────────────────────────
def aug_flip(rgb, v):
    return cv2.flip(rgb, [1, 0, -1][v])

def aug_rotate(rgb, v):
    angles = [15, -15, 30]
    h, w   = rgb.shape[:2]
    M      = cv2.getRotationMatrix2D((w/2, h/2), angles[v], 1.0)
    return cv2.warpAffine(rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def aug_zoom(rgb, v):
    pcts = [0.10, 0.20, 0.30]
    f    = pcts[v]; h, w = rgb.shape[:2]
    dh, dw = int(h*f/2), int(w*f/2)
    return cv2.resize(rgb[dh:h-dh, dw:w-dw], (w, h), interpolation=cv2.INTER_LINEAR)

def aug_blur(lab, v):
    return cv2.GaussianBlur(lab, (3, 3), [0.5, 1.0, 1.5][v])

def aug_noise(lab, v):
    rng = np.random.default_rng(SEED + v)
    n   = rng.normal(0, [3,6,9][v], lab.shape).astype(np.float32)
    return np.clip(lab.astype(np.float32) + n, 0, 255).astype(np.uint8)

AUG_CATALOGUE = [
    ('Flip',     aug_flip,   'rgb', 3),
    ('Rotation', aug_rotate, 'rgb', 3),
    ('Zoom',     aug_zoom,   'rgb', 3),
    ('Blur',     aug_blur,   'lab', 3),
    ('Noise',    aug_noise,  'lab', 3),
]
print(f'Augmentation variants per image: {sum(n for *_,n in AUG_CATALOGUE)}')
print(f'Training set multiplier: ×{1 + sum(n for *_,n in AUG_CATALOGUE)}')


In [ ]:
# ── Build augmented training set with multi-CS features and origin tracking ──
aug_feats_per_cs = {cs: [] for cs in COLOR_SPACES}
aug_feats_combined = []
aug_y_list = []
aug_orig_idx_list = []
aug_is_original_list = []

for i, (rgb, lab, lbl) in enumerate(tqdm(
        zip(X_rgb_tr_orig, X_lab_tr_orig, y_tr_orig),
        total=len(y_tr_orig), desc='Augmenting')):
    # Original
    res = stat_filter(rgb)
    all_f = extract_all_features(res['out'], res['kept_mask'])
    for cs in COLOR_SPACES:
        aug_feats_per_cs[cs].append(all_f[cs])
    aug_feats_combined.append(all_f['Combined'])
    aug_y_list.append(lbl)
    aug_orig_idx_list.append(i)
    aug_is_original_list.append(True)
    # Augmented variants
    for aug_name, aug_fn, domain, n_variants in AUG_CATALOGUE:
        for v in range(n_variants):
            if domain == 'rgb':
                aug_rgb = aug_fn(rgb, v)
                aug_lab_img = rgb_to_lab(aug_rgb)
            else:
                aug_lab_img = aug_fn(lab, v)
                aug_rgb = cv2.cvtColor(aug_lab_img, cv2.COLOR_LAB2RGB)
            res = stat_filter(aug_rgb)
            all_f = extract_all_features(res['out'], res['kept_mask'])
            for cs in COLOR_SPACES:
                aug_feats_per_cs[cs].append(all_f[cs])
            aug_feats_combined.append(all_f['Combined'])
            aug_y_list.append(lbl)
            aug_orig_idx_list.append(i)
            aug_is_original_list.append(False)

X_feats_tr_aug_per_cs = {cs: np.array(aug_feats_per_cs[cs], dtype=np.float32) for cs in COLOR_SPACES}
X_feats_tr_aug_combined = np.array(aug_feats_combined, dtype=np.float32)
X_feats_tr_aug = X_feats_tr_aug_per_cs['LAB']  # backward compatibility
y_tr_aug        = np.array(aug_y_list,     dtype=np.int32)
aug_orig_idx    = np.array(aug_orig_idx_list, dtype=np.int32)
aug_is_original = np.array(aug_is_original_list, dtype=bool)

print(f'Augmented training set: {X_feats_tr_aug.shape} (LAB)')
print(f'Combined features: {X_feats_tr_aug_combined.shape}')
print(f'Original samples: {aug_is_original.sum()}, Augmented: {(~aug_is_original).sum()}')

In [ ]:
# ── Augmentation grid (one row per score, one col per variant) ───────────────
aug_labels = (['Original'] +
              [f'Flip-{i+1}' for i in range(3)] +
              [f'Rot-{i+1}'  for i in range(3)] +
              [f'Zoom-{i+1}' for i in range(3)] +
              [f'Blur-{i+1}' for i in range(3)] +
              [f'Noise-{i+1}'for i in range(3)])
border_cols = (['#333']+['#2980b9']*3+['#27ae60']*3+['#8e44ad']*3+['#16a085']*3+['#c0392b']*3)
n_cols = len(aug_labels)
fig, axes = plt.subplots(NUM_CLASSES, n_cols, figsize=(n_cols*1.4, NUM_CLASSES*1.7))
for cls_i in range(NUM_CLASSES):
    idxs = np.where(y_tr_orig == cls_i+1)[0]
    if len(idxs) == 0: continue
    rgb0 = X_rgb_tr_orig[idxs[0]]; lab0 = X_lab_tr_orig[idxs[0]]
    display = [rgb0]
    for _, aug_fn, domain, nv in AUG_CATALOGUE:
        for v in range(nv):
            if domain == 'rgb':
                display.append(aug_fn(rgb0, v))
            else:
                display.append(cv2.cvtColor(aug_fn(lab0, v), cv2.COLOR_LAB2RGB))
    for col, (ax, img, bc) in enumerate(zip(axes[cls_i], display, border_cols)):
        ax.imshow(img)
        if cls_i == 0: ax.set_title(aug_labels[col], fontsize=5.5, color=bc, fontweight='bold')
        if col == 0:   ax.set_ylabel(f'S{cls_i+1}', fontsize=8, fontweight='bold')
        ax.axis('off')
        for sp in ax.spines.values(): sp.set_edgecolor(bc); sp.set_linewidth(1.5)
plt.suptitle('Colour-Safe Augmentation Grid per Score Class', fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout(); savefig('07_augmentation_grid.png'); plt.show()


## 8 · Feature Engineering — Model-Specific Pipelines

Each model uses a **different, scientifically motivated** feature engineering strategy.
The generic `build_feature_pipeline` is retained for shared utilities but each model
also has its own `build_<model>_pipeline` / `apply_<model>_pipeline` pair.

| Model | Input features | Why this combination | Scaling |
|---|---|---|---|
| **SVM (SVR-RBF)** | LAB (perceptual) + texture (range, IQR, p10, p90) + HSV-S (saturation) | RBF kernel is distance-based → needs standardised features; LAB is perceptually linear; texture captures surface non-uniformity | `StandardScaler` (mandatory for SVM) |
| **Random Forest** | HSV + LAB + full texture stats (all CS) | Ensembles of decision trees are scale-invariant; more features = better splits; HSV separates hue from brightness cleanly | **No scaling** (trees are scale-invariant) |
| **XGBoost** | LAB + HSV + XYZ + texture + histogram bins (all CS combined) | Gradient boosting gains from the broadest feature set; regularisation controls complexity; early-stopping prevents overfitting | **No scaling** (gradient boosting is scale-invariant) |

> **Modification rationale:** Previously all three models used the same LAB-only feature set with
> a StandardScaler applied uniformly. This is incorrect because:
> (1) RF and XGBoost are scale-invariant — scaling is harmless but wasteful and masks the benefit of
>     broader feature sets.
> (2) SVM *requires* scaling but benefits most from perceptually meaningful features.
> (3) XGBoost can exploit hundreds of features if regularisation is strong.
> We therefore use model-specific pipelines that play to each model's strengths.


In [ ]:
# ── Shared feature engineering utilities ─────────────────────────────────────
def correlation_filter(X: np.ndarray, threshold: float = 0.95) -> np.ndarray:
    """Return boolean mask of features to KEEP (drop higher-index of correlated pairs)."""
    corr   = np.corrcoef(X.T)
    n_feat = X.shape[1]
    keep   = np.ones(n_feat, dtype=bool)
    for i in range(n_feat):
        if not keep[i]: continue
        for j in range(i+1, n_feat):
            if keep[j] and abs(corr[i, j]) > threshold:
                keep[j] = False
    return keep


# Replace NaN in features
for cs in COLOR_SPACES:
    X_feats_tr_aug_per_cs[cs] = np.nan_to_num(X_feats_tr_aug_per_cs[cs], nan=0.0)
    X_feats_per_cs[cs]        = np.nan_to_num(X_feats_per_cs[cs],        nan=0.0)
X_feats_tr_aug_combined = np.nan_to_num(X_feats_tr_aug_combined, nan=0.0)
X_feats_combined        = np.nan_to_num(X_feats_combined,        nan=0.0)
X_feats_tr_aug          = np.nan_to_num(X_feats_tr_aug,          nan=0.0)
X_feats                 = np.nan_to_num(X_feats,                  nan=0.0)

# ── Generic pipeline (used for ablation / dim-reduction experiments) ──────────
def build_feature_pipeline(X_train, y_train, k_best: int, scale: bool = True):
    """
    Generic pipeline: VarianceThreshold → CorrelationFilter → SelectKBest → (StandardScaler).
    Fitted only on training data to prevent leakage.
    """
    X_train = np.nan_to_num(X_train, nan=0.0)
    vt    = VarianceThreshold(threshold=0.01).fit(X_train)
    Xv    = vt.transform(X_train)
    cmask = correlation_filter(Xv, 0.95)
    Xc    = Xv[:, cmask]
    k     = min(k_best, Xc.shape[1])
    kb    = SelectKBest(f_classif, k=k).fit(Xc, y_train)
    Xk    = kb.transform(Xc)
    sc    = StandardScaler().fit(Xk) if scale else None
    Xf    = sc.transform(Xk) if sc else Xk
    return Xf, (vt, cmask, kb, sc)


def apply_feature_pipeline(X, transformers):
    """Apply a pre-fitted pipeline to new data (val/test)."""
    vt, cmask, kb, sc = transformers
    X  = np.nan_to_num(X, nan=0.0)
    Xv = vt.transform(X)
    Xc = Xv[:, cmask]
    Xk = kb.transform(Xc)
    return sc.transform(Xk) if sc else Xk


# ── SVM-specific pipeline: LAB + texture + HSV-S, with StandardScaler ────────
# SVM (RBF kernel) is distance-based and requires standardised features.
# Features: all 72 LAB features + texture columns from HSV (S channel: range, IQR, p10, p90)
# + saturation mean for colour-saturation signal.
# Rationale: LAB is perceptually uniform → ideal distance metric; HSV-S adds
#            saturation information which LAB does not directly encode.

FEATS_PER_CH = 8 + N_HIST_BINS

def _svm_feature_indices():
    """Return column indices in the combined feature array for SVM input."""
    # LAB: all 72 features (indices 72..143 in combined array, after RGB)
    n_rgb = FEATS_PER_CH * 3   # 72 (skip RGB block)
    lab_start = n_rgb
    lab_end   = lab_start + FEATS_PER_CH * 3
    lab_idx   = list(range(lab_start, lab_end))

    # HSV-S: indices within HSV block (HSV block starts after RGB+LAB... wait:
    # Combined order = RGB, HSV, LAB, XYZ  (from COLOR_SPACES list)
    # So: RGB=0..71, HSV=72..143, LAB=144..215, XYZ=216..287
    hsv_start  = FEATS_PER_CH * 3        # 72
    s_ch_start = hsv_start + FEATS_PER_CH  # S is channel index 1 in HSV
    # texture cols within the channel: range=4, iqr=5, p10=6, p90=7  (0-indexed after mean,std,skew,kurt)
    texture_offsets = [4, 5, 6, 7]
    hsv_s_texture = [s_ch_start + o for o in texture_offsets]
    # HSV-S mean (index 0 of that channel)
    hsv_s_mean = [s_ch_start]

    return sorted(set(lab_idx + hsv_s_texture + hsv_s_mean))

_SVM_COLS = _svm_feature_indices()
print(f'SVM feature columns (LAB + HSV-S texture + HSV-S mean): {len(_SVM_COLS)}')


def build_svm_pipeline(X_tr_comb, y_tr):
    """
    SVM pipeline: select model-specific columns, variance-filter, corr-filter,
    SelectKBest (top 40), StandardScaler.
    """
    X = X_tr_comb[:, _SVM_COLS]
    X = np.nan_to_num(X, nan=0.0)
    vt    = VarianceThreshold(threshold=0.01).fit(X)
    Xv    = vt.transform(X)
    cmask = correlation_filter(Xv, 0.95)
    Xc    = Xv[:, cmask]
    k     = min(40, Xc.shape[1])
    kb    = SelectKBest(f_classif, k=k).fit(Xc, y_tr)
    Xk    = kb.transform(Xc)
    sc    = StandardScaler().fit(Xk)
    return sc.transform(Xk), (_SVM_COLS, vt, cmask, kb, sc)


def apply_svm_pipeline(X_comb, pipes):
    col_idx, vt, cmask, kb, sc = pipes
    X  = np.nan_to_num(X_comb[:, col_idx], nan=0.0)
    Xv = vt.transform(X)
    Xc = Xv[:, cmask]
    Xk = kb.transform(Xc)
    return sc.transform(Xk)


# ── RF-specific pipeline: HSV + LAB + all texture, NO scaling ─────────────────
# Random Forest is scale-invariant; more features → more potential split candidates.
# We use HSV + LAB (144 features) to focus on colour and appearance without XYZ redundancy.
# XYZ is excluded here (tested in ablation; see Step 14) because X and Z are linear
# combinations of R,G,B and add little over HSV+LAB for tree-based splits.
# No SelectKBest: RF internal feature selection via max_features handles this.

def _rf_feature_indices():
    """HSV block (72) + LAB block (72) = 144 features."""
    hsv_start = FEATS_PER_CH * 3          # 72 (after RGB)
    lab_start = FEATS_PER_CH * 3 * 2      # 144 (after RGB+HSV)
    return list(range(hsv_start, hsv_start + FEATS_PER_CH * 3)) +            list(range(lab_start, lab_start + FEATS_PER_CH * 3))

_RF_COLS = _rf_feature_indices()
print(f'RF feature columns (HSV + LAB, no scaling): {len(_RF_COLS)}')


def build_rf_pipeline(X_tr_comb, y_tr):
    """RF pipeline: select HSV+LAB columns, variance-filter, corr-filter. No scaling."""
    X = np.nan_to_num(X_tr_comb[:, _RF_COLS], nan=0.0)
    vt    = VarianceThreshold(threshold=0.01).fit(X)
    Xv    = vt.transform(X)
    cmask = correlation_filter(Xv, 0.95)
    return Xv[:, cmask], (_RF_COLS, vt, cmask)


def apply_rf_pipeline(X_comb, pipes):
    col_idx, vt, cmask = pipes
    X  = np.nan_to_num(X_comb[:, col_idx], nan=0.0)
    Xv = vt.transform(X)
    return Xv[:, cmask]


# ── XGBoost-specific pipeline: ALL color spaces + texture, NO scaling ─────────
# XGBoost can exploit a very wide feature set when regularisation is strong.
# We use the full combined array (288 features) and let variance + correlation
# filtering reduce obvious redundancy. XGBoost's own feature sub-sampling
# (colsample_bytree) then handles further selection internally.

def build_xgb_pipeline(X_tr_comb, y_tr):
    """XGBoost pipeline: full combined features, variance+corr filter, no scaling."""
    X = np.nan_to_num(X_tr_comb, nan=0.0)
    vt    = VarianceThreshold(threshold=0.01).fit(X)
    Xv    = vt.transform(X)
    cmask = correlation_filter(Xv, 0.95)
    return Xv[:, cmask], (vt, cmask)


def apply_xgb_pipeline(X_comb, pipes):
    vt, cmask = pipes
    X  = np.nan_to_num(X_comb, nan=0.0)
    Xv = vt.transform(X)
    return Xv[:, cmask]


# Pre-compute test set combined features for reuse
X_feats_te          = np.nan_to_num(X_feats_per_cs['LAB'][idx_te],    nan=0.0)
X_feats_te_combined = np.nan_to_num(X_feats_combined[idx_te],           nan=0.0)

print(f'\nLAB train aug  : {X_feats_tr_aug.shape}')
print(f'Combined train : {X_feats_tr_aug_combined.shape}')
print(f'Pipeline helpers ready ✔ (SVM / RF / XGBoost)')


## 9 · Model Training & Hyperparameter Tuning

### Strategy

- **5-Fold Stratified CV** on augmented training set (train on 4 folds, validate on 5th)
- **Optuna** (100 trials per model, TPE sampler + MedianPruner) optimises a composite objective
- **Composite objective:** 0.50·ρ + 0.25·R² + 0.25·Acc@1
- **Three models**: SVM, Random Forest, XGBoost — each with its own feature pipeline

### Model-specific pipelines trained in this section

| Section | Model | Feature set | Scaling | Pipelines compared |
|---|---|---|---|---|
| 9.1 | **SVM (SVR-RBF)** | LAB + texture + HSV-S | StandardScaler | (A) model-specific; (B) generic LAB only |
| 9.2 | **Random Forest** | HSV + LAB | None | (A) model-specific; (B) full combined |
| 9.3 | **XGBoost** | All CS combined | None | (A) model-specific; (B) LAB-only baseline |

> **KNN removed:** KNN (k-nearest-neighbours) was included in the previous version but removed here because:
> (1) KNN is a lazy learner with no explicit feature weighting — it is highly sensitive to the curse of dimensionality with 72–288 features.
> (2) In preliminary experiments (available in v1 results), KNN consistently underperformed SVR and RF by a large margin.
> (3) KNN provides no feature importance information, which is a key deliverable of this notebook.
> (4) Computational cost of KNN prediction scales with training set size (slow at inference).

### Overfitting controls (applied to all models)
- CV validates on **original (non-augmented)** images only to prevent leakage
- `overfit_diagnostic()` prints train vs val composite gap per fold
- XGBoost: early stopping (patience=40) + depth cap + strong L1/L2 regularisation


In [ ]:
# ── Shared evaluation helper ─────────────────────────────────────────────────
def evaluate(y_true: np.ndarray, y_pred_cont: np.ndarray) -> dict:
    """Compute all metrics given continuous predictions and integer true scores."""
    y_pred_r = np.clip(np.round(y_pred_cont), 1, 8).astype(int)
    y_true   = np.array(y_true, dtype=int)
    mae  = mean_absolute_error(y_true, y_pred_cont)
    mse  = mean_squared_error(y_true, y_pred_cont)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred_cont)
    acc  = float(np.mean(y_pred_r == y_true))
    rho, _  = spearmanr(y_true, y_pred_cont)
    tau, _  = kendalltau(y_true, y_pred_cont)
    acc1 = float(np.mean(np.abs(y_pred_r - y_true) <= 1))
    cm   = confusion_matrix(y_true, y_pred_r, labels=list(range(1, 9)))
    return {'MAE':mae,'MSE':mse,'RMSE':rmse,'R2':r2,'Accuracy':acc,
            'Spearman_rho':rho,'Kendall_tau':tau,'Acc@0':acc,'Acc@1':acc1,'CM':cm}



# ── Augmentation-aware 5-fold cross-validation ───────────────────────────────
# K=5 → train on 4 folds (80%), validate on the held-out 5th fold (20%).
# Validation uses ONLY original (non-augmented) samples to prevent leakage.
# Returns both CV scores AND per-fold train/val composite for overfitting detection.

def cv_objective(trial, model_fn, X, y, k_best=20, return_fold_detail=False, scale=True):
    _cv_scale = scale
    """
    5-fold stratified CV — train on folds 0..3, validate on fold 4 (rotated each time).
    Each fold: train = 4 folds of augmented data; val = 1 fold of ORIGINAL images only.
    Composite objective: 0.50·ρ + 0.25·R² + 0.25·Acc@1
    Anti-overfit guard: also computes TRAIN composite per fold and prints gap when
    return_fold_detail=True.
    """
    assert N_FOLDS == 5, "cv_objective assumes 5-fold CV"
    n_orig = int(aug_orig_idx.max()) + 1
    y_orig_labels = y[aug_is_original]
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    val_scores  = []
    train_scores = []

    for fold_i, (tr_orig, va_orig) in enumerate(skf.split(np.arange(n_orig), y_orig_labels)):
        # tr_orig: indices of 4 training folds → va_orig: 1 validation fold
        tr_mask = np.isin(aug_orig_idx, tr_orig)           # includes all augmented copies
        va_mask = np.isin(aug_orig_idx, va_orig) & aug_is_original  # originals only

        X_tr, y_tr = X[tr_mask], y[tr_mask]
        X_va, y_va = X[va_mask], y[va_mask]

        Xf_tr, pipes = build_feature_pipeline(X_tr, y_tr, k_best, scale=_cv_scale)
        Xf_va        = apply_feature_pipeline(X_va, pipes)

        model = model_fn(trial)
        model.fit(Xf_tr, y_tr)

        # --- Validation composite (main objective) ---
        preds_va = np.clip(model.predict(Xf_va), 1, 8)
        rho_va, _ = spearmanr(y_va, preds_va)
        r2_va     = r2_score(y_va, preds_va)
        acc1_va   = float(np.mean(np.abs(np.clip(np.round(preds_va),1,8).astype(int) - y_va) <= 1))
        comp_va   = 0.50 * max(0, rho_va) + 0.25 * max(0, r2_va) + 0.25 * acc1_va
        val_scores.append(comp_va)

        # --- Train composite (for overfitting diagnostic) ---
        if return_fold_detail:
            # Sample originals only from training fold for a fair comparison
            tr_orig_mask = np.isin(aug_orig_idx, tr_orig) & aug_is_original
            Xf_tr_orig   = apply_feature_pipeline(X[tr_orig_mask], pipes)  # uses fitted pipes
            preds_tr     = np.clip(model.predict(Xf_tr_orig), 1, 8)
            y_tr_orig_v  = y[tr_orig_mask]
            rho_tr, _    = spearmanr(y_tr_orig_v, preds_tr)
            r2_tr        = r2_score(y_tr_orig_v, preds_tr)
            acc1_tr      = float(np.mean(np.abs(np.clip(np.round(preds_tr),1,8).astype(int) - y_tr_orig_v) <= 1))
            comp_tr      = 0.50 * max(0, rho_tr) + 0.25 * max(0, r2_tr) + 0.25 * acc1_tr
            train_scores.append(comp_tr)

    mean_val = float(np.mean(val_scores)) if val_scores else 0.0
    if return_fold_detail:
        return mean_val, val_scores, train_scores
    return mean_val


# ── Quick overfitting diagnostic (used post-tuning for each best model) ───────
def overfit_diagnostic(model_name, make_model, X_aug, y_aug, k_best):
    """Train 5-fold and report train vs val composite gap per fold."""
    class _DummyTrial:
        """Minimal trial stub so model_fn(trial) works with tuned params."""
        pass

    dummy_trial = _DummyTrial()
    mean_val, val_scores, train_scores = cv_objective(
        dummy_trial, lambda _: make_model(), X_aug, y_aug,
        k_best=k_best, return_fold_detail=True)

    print(f"\n📊 Overfitting diagnostic — {model_name} (k_best={k_best})")
    print(f"  {'Fold':<6} {'Train composite':>16} {'Val composite':>14} {'Gap (train-val)':>16}")
    for i, (tr, va) in enumerate(zip(train_scores, val_scores)):
        gap = tr - va
        flag = '⚠️ overfit' if gap > 0.20 else ''
        print(f"  {i+1:<6} {tr:>16.4f} {va:>14.4f} {gap:>16.4f}  {flag}")
    print(f"  {'Mean':<6} {np.mean(train_scores):>16.4f} {np.mean(val_scores):>14.4f} "
          f"{np.mean(train_scores)-np.mean(val_scores):>16.4f}")


results = {}
print('Helpers ready ✔')
print(f'  5-fold CV: train on 4 folds, validate on 1 fold (original images only)')
print(f'  Composite objective: 0.50·ρ + 0.25·R² + 0.25·Acc@1')
print(f'  Overfitting diagnostic: train vs val composite gap printed per model')


### 9.1 · SVM — SVR with RBF Kernel

**Pipeline A (model-specific):** LAB (perceptual, 72 features) + HSV-S texture + HSV-S mean  
→ SelectKBest (top 40) → StandardScaler → SVR-RBF

**Pipeline B (generic LAB baseline):** LAB only → SelectKBest → StandardScaler → SVR-RBF

**Rationale for LAB:** The RBF kernel measures Euclidean distances in feature space. LAB is
a perceptually uniform space where Euclidean distance ≈ perceived colour difference (ΔE),
making it the ideal space for a distance-based kernel. HSV saturation is added because it
captures colour purity independently of lightness, which is not directly encoded in LAB.

**Rationale for StandardScaler:** Mandatory for SVM — features must be on comparable scales
for the RBF bandwidth parameter (gamma) to behave uniformly across all dimensions.


In [ ]:
# ── 9.1 SVM: Pipeline A (LAB + HSV-S texture) and Pipeline B (LAB only) ──────

# ─── Pipeline A: model-specific ──────────────────────────────────────────────
print('=' * 60)
print('SVM Pipeline A: LAB + HSV-S texture + StandardScaler')
print('=' * 60)

def tune_svr_A():
    # Pre-build pipeline once outside the loop for efficiency
    # (feature selection k is still tuned inside Optuna)
    def objective(trial):
        k = trial.suggest_int('k_best', 10, 50)
        C       = trial.suggest_float('C',       0.01, 300.0, log=True)
        gamma   = trial.suggest_float('gamma',   1e-5, 2.0,   log=True)
        epsilon = trial.suggest_float('epsilon', 0.005, 1.5)

        # Build feature matrix on the fly per fold inside cv_objective
        def model_fn(t):
            return SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon,
                       max_iter=15000, cache_size=500)

        # Use SVM-specific columns from combined array
        X_svm_aug = X_feats_tr_aug_combined[:, _SVM_COLS]
        return cv_objective(trial, model_fn, X_svm_aug, y_tr_aug,
                            k_best=k, scale=True)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

t0 = time.time()
best_svr_A = tune_svr_A()
train_time_svr_A = time.time() - t0
print(f'Tuning done in {train_time_svr_A:.1f}s. Best params: {best_svr_A}')

# Final fit
X_svm_aug = X_feats_tr_aug_combined[:, _SVM_COLS]
X_svm_te  = X_feats_te_combined[:, _SVM_COLS]
Xf_tr_sA, pipes_sA = build_feature_pipeline(X_svm_aug, y_tr_aug,
                                              k_best=best_svr_A['k_best'], scale=True)
Xf_te_sA = apply_feature_pipeline(X_svm_te, pipes_sA)
model_svr_A = SVR(kernel='rbf', C=best_svr_A['C'], gamma=best_svr_A['gamma'],
                  epsilon=best_svr_A['epsilon'], max_iter=15000, cache_size=500)
model_svr_A.fit(Xf_tr_sA, y_tr_aug)
preds_svr_A = np.clip(model_svr_A.predict(Xf_te_sA), 1, 8)
results['SVM-A (LAB+HSV-S+texture)'] = {
    'metrics': evaluate(y_te, preds_svr_A),
    'best_params': best_svr_A, 'preds': preds_svr_A,
    'train_time': train_time_svr_A,
    'feature_set': 'LAB + HSV-S texture',
    'color_space': 'LAB+HSV',
}
m = results['SVM-A (LAB+HSV-S+texture)']['metrics']
print(f"  R²={m['R2']:.4f}  RMSE={m['RMSE']:.3f}  MAE={m['MAE']:.3f}  ρ={m['Spearman_rho']:.4f}")

overfit_diagnostic('SVM-A',
    lambda: SVR(kernel='rbf', C=best_svr_A['C'], gamma=best_svr_A['gamma'],
                epsilon=best_svr_A['epsilon'], max_iter=15000, cache_size=500),
    X_svm_aug, y_tr_aug, best_svr_A['k_best'])


# ─── Pipeline B: LAB-only baseline ───────────────────────────────────────────
print('\n' + '=' * 60)
print('SVM Pipeline B: LAB only (baseline) + StandardScaler')
print('=' * 60)

def tune_svr_B():
    def objective(trial):
        k       = trial.suggest_int('k_best', 10, 40)
        C       = trial.suggest_float('C',       0.01, 300.0, log=True)
        gamma   = trial.suggest_float('gamma',   1e-5, 2.0,   log=True)
        epsilon = trial.suggest_float('epsilon', 0.005, 1.5)
        def model_fn(t):
            return SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon,
                       max_iter=15000, cache_size=500)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug,
                            k_best=k, scale=True)
    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

t0 = time.time()
best_svr_B = tune_svr_B()
train_time_svr_B = time.time() - t0
print(f'Tuning done in {train_time_svr_B:.1f}s. Best params: {best_svr_B}')

Xf_tr_sB, pipes_sB = build_feature_pipeline(X_feats_tr_aug, y_tr_aug,
                                              k_best=best_svr_B['k_best'], scale=True)
Xf_te_sB = apply_feature_pipeline(X_feats_te, pipes_sB)
model_svr_B = SVR(kernel='rbf', C=best_svr_B['C'], gamma=best_svr_B['gamma'],
                  epsilon=best_svr_B['epsilon'], max_iter=15000, cache_size=500)
model_svr_B.fit(Xf_tr_sB, y_tr_aug)
preds_svr_B = np.clip(model_svr_B.predict(Xf_te_sB), 1, 8)
results['SVM-B (LAB only)'] = {
    'metrics': evaluate(y_te, preds_svr_B),
    'best_params': best_svr_B, 'preds': preds_svr_B,
    'train_time': train_time_svr_B,
    'feature_set': 'LAB only',
    'color_space': 'LAB',
}
m = results['SVM-B (LAB only)']['metrics']
print(f"  R²={m['R2']:.4f}  RMSE={m['RMSE']:.3f}  MAE={m['MAE']:.3f}  ρ={m['Spearman_rho']:.4f}")
print(f"\n  → SVM-A vs SVM-B R² gap: {results['SVM-A (LAB+HSV-S+texture)']['metrics']['R2'] - m['R2']:+.4f}")
print(f"    Positive → adding HSV-S texture features helped.")


### 9.2 · Random Forest

**Pipeline A (model-specific):** HSV + LAB (144 features) → variance+corr filter → RF (no scaling)  
**Pipeline B (generic combined):** All 4 CS combined (288 features) → variance+corr filter → RF

**Rationale for no scaling:** Decision trees split on feature thresholds, not distances —
scaling has zero effect on tree structure. Applying StandardScaler to RF would add unnecessary
computation without any benefit.

**Rationale for HSV + LAB:** HSV provides hue separation (H) independent of brightness,
saturation (S), and a direct brightness channel (V). LAB adds perceptual lightness (L*) and
chromatic axes. Together they avoid the strong channel correlations in RGB and XYZ.
XYZ is excluded because X = 0.41R + 0.36G + 0.18B and Z ≈ 1.05B — these are linear
combinations of RGB that add minimal new information over HSV+LAB for tree split criteria.


In [ ]:
# ── 9.2 Random Forest: Pipeline A (HSV+LAB) and Pipeline B (All CS combined) ─

# ─── Pipeline A: HSV + LAB, no scaling ───────────────────────────────────────
print('=' * 60)
print('RF Pipeline A: HSV + LAB features, no scaling')
print('=' * 60)

def tune_rf_A():
    def objective(trial):
        n_est = trial.suggest_int('n_estimators', 100, 600)
        depth = trial.suggest_int('max_depth', 4, 20)
        msl   = trial.suggest_int('min_samples_leaf', 1, 15)
        mss   = trial.suggest_int('min_samples_split', 2, 20)
        mf    = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5])
        ccp   = trial.suggest_float('ccp_alpha', 0.0, 0.05)
        def model_fn(t):
            return RandomForestRegressor(n_estimators=n_est, max_depth=depth,
                                         min_samples_leaf=msl, min_samples_split=mss,
                                         max_features=mf, ccp_alpha=ccp,
                                         random_state=SEED, n_jobs=-1)
        # Pass full combined array — build_rf_pipeline slices _RF_COLS internally
        return cv_objective(trial, model_fn, X_feats_tr_aug_combined, y_tr_aug,
                            k_best=min(60, len(_RF_COLS)), scale=False)

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

t0 = time.time()
best_rf_A = tune_rf_A()
train_time_rf_A = time.time() - t0
print(f'Tuning done in {train_time_rf_A:.1f}s. Best params: {best_rf_A}')

# Pass full combined arrays — build_rf_pipeline / apply_rf_pipeline slice _RF_COLS internally
Xf_tr_rA, pipes_rA = build_rf_pipeline(X_feats_tr_aug_combined, y_tr_aug)
Xf_te_rA = apply_rf_pipeline(X_feats_te_combined, pipes_rA)
rf_params_A = {k: v for k, v in best_rf_A.items()}
model_rf_A = RandomForestRegressor(**rf_params_A, random_state=SEED, n_jobs=-1)
model_rf_A.fit(Xf_tr_rA, y_tr_aug)
preds_rf_A = np.clip(model_rf_A.predict(Xf_te_rA), 1, 8)
results['RF-A (HSV+LAB)'] = {
    'metrics': evaluate(y_te, preds_rf_A),
    'best_params': best_rf_A, 'preds': preds_rf_A,
    'train_time': train_time_rf_A,
    'feature_set': 'HSV + LAB',
    'color_space': 'HSV+LAB',
}
m = results['RF-A (HSV+LAB)']['metrics']
print(f"  R²={m['R2']:.4f}  RMSE={m['RMSE']:.3f}  MAE={m['MAE']:.3f}  ρ={m['Spearman_rho']:.4f}")

overfit_diagnostic('RF-A (HSV+LAB)',
    lambda: RandomForestRegressor(**{k: v for k,v in best_rf_A.items()},
                                   random_state=SEED, n_jobs=-1),
    X_feats_tr_aug_combined, y_tr_aug, min(60, Xf_tr_rA.shape[1]))


# ─── Pipeline B: all CS combined ─────────────────────────────────────────────
print('\n' + '=' * 60)
print('RF Pipeline B: All 4 CS combined (288 features), no scaling')
print('=' * 60)

def tune_rf_B():
    def objective(trial):
        k     = trial.suggest_int('k_best', 20, 80)
        n_est = trial.suggest_int('n_estimators', 100, 600)
        depth = trial.suggest_int('max_depth', 4, 20)
        msl   = trial.suggest_int('min_samples_leaf', 1, 15)
        mss   = trial.suggest_int('min_samples_split', 2, 20)
        mf    = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5])
        ccp   = trial.suggest_float('ccp_alpha', 0.0, 0.05)
        def model_fn(t):
            return RandomForestRegressor(n_estimators=n_est, max_depth=depth,
                                         min_samples_leaf=msl, min_samples_split=mss,
                                         max_features=mf, ccp_alpha=ccp,
                                         random_state=SEED, n_jobs=-1)
        return cv_objective(trial, model_fn, X_feats_tr_aug_combined, y_tr_aug,
                            k_best=k, scale=False)
    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED),
                                pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

t0 = time.time()
best_rf_B = tune_rf_B()
train_time_rf_B = time.time() - t0
print(f'Tuning done in {train_time_rf_B:.1f}s. Best params: {best_rf_B}')

k_rfB = best_rf_B.pop('k_best')
Xf_tr_rB, pipes_rB = build_feature_pipeline(X_feats_tr_aug_combined, y_tr_aug,
                                              k_best=k_rfB, scale=False)
Xf_te_rB = apply_feature_pipeline(X_feats_te_combined, pipes_rB)
model_rf_B = RandomForestRegressor(**best_rf_B, random_state=SEED, n_jobs=-1)
model_rf_B.fit(Xf_tr_rB, y_tr_aug)
preds_rf_B = np.clip(model_rf_B.predict(Xf_te_rB), 1, 8)
results['RF-B (All CS)'] = {
    'metrics': evaluate(y_te, preds_rf_B),
    'best_params': {**best_rf_B, 'k_best': k_rfB}, 'preds': preds_rf_B,
    'train_time': train_time_rf_B,
    'feature_set': 'All 4 CS combined',
    'color_space': 'RGB+HSV+LAB+XYZ',
}
m = results['RF-B (All CS)']['metrics']
print(f"  R²={m['R2']:.4f}  RMSE={m['RMSE']:.3f}  MAE={m['MAE']:.3f}  ρ={m['Spearman_rho']:.4f}")
print(f"  → RF-A vs RF-B R² gap: {results['RF-A (HSV+LAB)']['metrics']['R2'] - m['R2']:+.4f}")
print(f"    Negative → broader feature set helped; Positive → focused features better.")


### 9.3 · XGBoost

**Pipeline A (model-specific):** All 4 CS combined (288 → variance+corr filtered) → XGBoost  
**Pipeline B (LAB-only baseline):** LAB (72 features) → variance+corr filter → XGBoost

**Rationale for full combined features:** XGBoost builds additive trees sequentially;
with strong `colsample_bytree` sub-sampling, it effectively performs feature selection
internally at each tree. A broader input feature set gives it more candidates to choose from.
Including XYZ allows the model to discover if luminance (Y) or chromaticity (X,Z) provide
signal orthogonal to LAB and HSV. If XYZ features never appear in top importances, this
is documented in Step 11.

**Anti-overfitting controls:**
- `max_depth` ≤ 5 (shallow trees)
- `min_child_weight` 5–20 (minimum samples per leaf)
- `reg_lambda` 5–200 (L2), `reg_alpha` 0.1–50 (L1)
- `gamma` ≥ 1.0 (split complexity penalty)
- `subsample` 0.5–0.8 (row sub-sampling)
- `colsample_bytree` 0.4–0.75 (column sub-sampling)
- Early stopping (patience=40) on a held-out 15% slice


In [ ]:
# ── 9.3 XGBoost: Pipeline A (All CS combined) and Pipeline B (LAB baseline) ──

# ─── Pipeline A: All CS combined ─────────────────────────────────────────────
print('=' * 60)
print('XGBoost Pipeline A: All 4 CS combined, no scaling')
print('=' * 60)

def tune_xgb_A():
    def objective(trial):
        n_est = trial.suggest_int('n_estimators', 100, 600)
        depth = trial.suggest_int('max_depth', 2, 5)
        lr    = trial.suggest_float('learning_rate', 0.005, 0.10, log=True)
        sub   = trial.suggest_float('subsample', 0.50, 0.80)
        col   = trial.suggest_float('colsample_bytree', 0.40, 0.75)
        lmbd  = trial.suggest_float('reg_lambda', 5.0, 200.0, log=True)
        alpha = trial.suggest_float('reg_alpha',  0.10,  50.0, log=True)
        mcw   = trial.suggest_int('min_child_weight', 5, 20)
        gamma = trial.suggest_float('gamma', 1.0, 20.0, log=True)
        def model_fn(t):
            return xgb.XGBRegressor(n_estimators=n_est, max_depth=depth,
                                     learning_rate=lr, subsample=sub,
                                     colsample_bytree=col, reg_lambda=lmbd,
                                     reg_alpha=alpha, min_child_weight=mcw,
                                     gamma=gamma, random_state=SEED, n_jobs=-1,
                                     eval_metric='rmse', verbosity=0)
        Xf_xgb, _ = build_xgb_pipeline(X_feats_tr_aug_combined, y_tr_aug)
        return cv_objective(trial, model_fn, Xf_xgb, y_tr_aug,
                            k_best=Xf_xgb.shape[1], scale=False)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=3))
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

t0 = time.time()
best_xgb_A = tune_xgb_A()
train_time_xgb_A = time.time() - t0
print(f'Tuning done in {train_time_xgb_A:.1f}s. Best params: {best_xgb_A}')

Xf_tr_xA, pipes_xA = build_xgb_pipeline(X_feats_tr_aug_combined, y_tr_aug)
Xf_te_xA            = apply_xgb_pipeline(X_feats_te_combined, pipes_xA)
# Final fit with early stopping on 15% holdout
from sklearn.model_selection import train_test_split as _tts
_Xtr_x, _Xval_x, _ytr_x, _yval_x = _tts(Xf_tr_xA, y_tr_aug, test_size=0.15, random_state=SEED)
xgb_params_A = dict(best_xgb_A)
model_xgb_A = xgb.XGBRegressor(**xgb_params_A, random_state=SEED, n_jobs=-1,
                                 eval_metric='rmse', verbosity=0,
                                 early_stopping_rounds=40)
model_xgb_A.fit(_Xtr_x, _ytr_x, eval_set=[(_Xval_x, _yval_x)], verbose=False)
print(f'  Early stopping: best_iteration={model_xgb_A.best_iteration}')

preds_xgb_A = np.clip(model_xgb_A.predict(Xf_te_xA), 1, 8)
results['XGB-A (All CS)'] = {
    'metrics': evaluate(y_te, preds_xgb_A),
    'best_params': xgb_params_A, 'preds': preds_xgb_A,
    'train_time': train_time_xgb_A,
    'feature_set': 'All 4 CS combined',
    'color_space': 'RGB+HSV+LAB+XYZ',
}
m = results['XGB-A (All CS)']['metrics']
print(f"  R²={m['R2']:.4f}  RMSE={m['RMSE']:.3f}  MAE={m['MAE']:.3f}  ρ={m['Spearman_rho']:.4f}")

overfit_diagnostic('XGB-A',
    lambda: xgb.XGBRegressor(**xgb_params_A, random_state=SEED, n_jobs=-1,
                               eval_metric='rmse', verbosity=0),
    Xf_tr_xA, y_tr_aug, Xf_tr_xA.shape[1])


# ─── Pipeline B: LAB-only baseline ───────────────────────────────────────────
print('\n' + '=' * 60)
print('XGBoost Pipeline B: LAB only (72 features), no scaling')
print('=' * 60)

def tune_xgb_B():
    def objective(trial):
        n_est = trial.suggest_int('n_estimators', 100, 600)
        depth = trial.suggest_int('max_depth', 2, 5)
        lr    = trial.suggest_float('learning_rate', 0.005, 0.10, log=True)
        sub   = trial.suggest_float('subsample', 0.50, 0.80)
        col   = trial.suggest_float('colsample_bytree', 0.40, 0.75)
        lmbd  = trial.suggest_float('reg_lambda', 5.0, 200.0, log=True)
        alpha = trial.suggest_float('reg_alpha',  0.10,  50.0, log=True)
        mcw   = trial.suggest_int('min_child_weight', 5, 20)
        gamma = trial.suggest_float('gamma', 1.0, 20.0, log=True)
        def model_fn(t):
            return xgb.XGBRegressor(n_estimators=n_est, max_depth=depth,
                                     learning_rate=lr, subsample=sub,
                                     colsample_bytree=col, reg_lambda=lmbd,
                                     reg_alpha=alpha, min_child_weight=mcw,
                                     gamma=gamma, random_state=SEED, n_jobs=-1,
                                     eval_metric='rmse', verbosity=0)
        return cv_objective(trial, model_fn, X_feats_tr_aug, y_tr_aug,
                            k_best=min(40, X_feats_tr_aug.shape[1]), scale=False)
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=3))
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
    return study.best_params

t0 = time.time()
best_xgb_B = tune_xgb_B()
train_time_xgb_B = time.time() - t0
print(f'Tuning done in {train_time_xgb_B:.1f}s. Best params: {best_xgb_B}')

Xf_tr_xB, pipes_xB = build_xgb_pipeline(X_feats_tr_aug, y_tr_aug)
Xf_te_xB            = apply_xgb_pipeline(X_feats_te, pipes_xB)
xgb_params_B = dict(best_xgb_B)
_Xtr_xb, _Xval_xb, _ytr_xb, _yval_xb = _tts(Xf_tr_xB, y_tr_aug, test_size=0.15, random_state=SEED)
model_xgb_B = xgb.XGBRegressor(**xgb_params_B, random_state=SEED, n_jobs=-1,
                                 eval_metric='rmse', verbosity=0,
                                 early_stopping_rounds=40)
model_xgb_B.fit(_Xtr_xb, _ytr_xb, eval_set=[(_Xval_xb, _yval_xb)], verbose=False)
print(f'  Early stopping: best_iteration={model_xgb_B.best_iteration}')

preds_xgb_B = np.clip(model_xgb_B.predict(Xf_te_xB), 1, 8)
results['XGB-B (LAB only)'] = {
    'metrics': evaluate(y_te, preds_xgb_B),
    'best_params': xgb_params_B, 'preds': preds_xgb_B,
    'train_time': train_time_xgb_B,
    'feature_set': 'LAB only',
    'color_space': 'LAB',
}
m = results['XGB-B (LAB only)']['metrics']
print(f"  R²={m['R2']:.4f}  RMSE={m['RMSE']:.3f}  MAE={m['MAE']:.3f}  ρ={m['Spearman_rho']:.4f}")
print(f"  → XGB-A vs XGB-B R² gap: {results['XGB-A (All CS)']['metrics']['R2'] - m['R2']:+.4f}")
print(f"    Positive → adding RGB+HSV+XYZ over LAB alone helped for XGBoost.")


### 9.4 · Summary — All Pipelines Trained

The `results` dict now contains six entries:

| Key | Model | Feature set | Pipeline |
|---|---|---|---|
| `SVM-A` | SVR-RBF | LAB + HSV-S texture | Model-specific |
| `SVM-B` | SVR-RBF | LAB only | Baseline |
| `RF-A`  | RF | HSV + LAB | Model-specific |
| `RF-B`  | RF | All 4 CS | Broader comparison |
| `XGB-A` | XGBoost | All 4 CS combined | Model-specific |
| `XGB-B` | XGBoost | LAB only | Baseline |

> These six pipelines are compared side-by-side in Section 10 (evaluation)
> and consolidated in Section 14 (final comparison table).


In [ ]:
# ── 9.5 · XYZ Color Space Investigation ──────────────────────────────────────
# REQ 3: Investigate whether XYZ contributes useful information vs RGB/HSV/LAB.
# Approach: (a) channel correlations with score, (b) standalone XYZ model,
#           (c) ablation: Combined-without-XYZ vs Full combined

FEATS_PER_CH = 8 + N_HIST_BINS
from scipy.stats import pearsonr

print('📊 XYZ Channel Correlation with Score')
print(f'{"Channel":<18} {"Spearman ρ":>12} {"Kendall τ":>12} {"Pearson r":>12}')
print('-' * 58)
for cs_label, cs in [('XYZ', 'XYZ'), ('LAB', 'LAB')]:
    for ch_i, ch in enumerate(CS_CHANNEL_NAMES[cs]):
        feat_mean = X_feats_per_cs[cs][:, ch_i * FEATS_PER_CH]
        rho_v, _ = spearmanr(y_raw, feat_mean)
        tau_v, _ = kendalltau(y_raw, feat_mean)
        r_v, _   = pearsonr(y_raw.astype(float), feat_mean.astype(float))
        print(f'  {cs_label} {ch:<12}  {rho_v:>12.4f}  {tau_v:>12.4f}  {r_v:>12.4f}')
    print()

print('📊 Cross-correlation: XYZ vs LAB channel means (Pearson r)')
xyz_chs = CS_CHANNEL_NAMES['XYZ']; lab_chs = CS_CHANNEL_NAMES['LAB']
print(f'{"":20}' + ''.join([f'{c:>10}' for c in lab_chs]))
for ci, chx in enumerate(xyz_chs):
    xyz_m = X_feats_per_cs['XYZ'][:, ci * FEATS_PER_CH].astype(float)
    row = f'  XYZ {chx:<14}'
    for cj in range(3):
        lab_m = X_feats_per_cs['LAB'][:, cj * FEATS_PER_CH].astype(float)
        r, _ = pearsonr(xyz_m, lab_m); row += f'  {r:>8.4f}'
    print(row)

# Standalone XYZ RF
print('\n📊 Standalone XYZ model (RF)')
X_xyz_aug  = X_feats_tr_aug_per_cs['XYZ']
X_xyz_te_s   = np.nan_to_num(X_feats_per_cs['XYZ'][idx_te], nan=0.0)
Xf_xyz_tr_s, pipes_xyz_s = build_feature_pipeline(X_xyz_aug, y_tr_aug, k_best=30, scale=False)
Xf_xyz_te_s  = apply_feature_pipeline(X_xyz_te_s, pipes_xyz_s)
rf_xyz_m   = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1)
rf_xyz_m.fit(Xf_xyz_tr_s, y_tr_aug)
p_xyz      = np.clip(rf_xyz_m.predict(Xf_xyz_te_s), 1, 8)
m_xyz      = evaluate(y_te, p_xyz)
print(f'  XYZ alone:  R²={m_xyz["R2"]:.4f}  RMSE={m_xyz["RMSE"]:.3f}  rho={m_xyz["Spearman_rho"]:.4f}')

# No-XYZ ablation: RF on RGB+HSV+LAB only (first 216 cols of combined)
xyz_start    = FEATS_PER_CH * 3 * 3   # 216
no_xyz_cols  = list(range(0, xyz_start))
Xf_noxyz_tr, pipes_noxyz = build_feature_pipeline(
    X_feats_tr_aug_combined[:, no_xyz_cols], y_tr_aug, k_best=60, scale=False)
Xf_noxyz_te = apply_feature_pipeline(X_feats_te_combined[:, no_xyz_cols], pipes_noxyz)
rf_noxyz_m  = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=SEED, n_jobs=-1)
rf_noxyz_m.fit(Xf_noxyz_tr, y_tr_aug)
p_noxyz     = np.clip(rf_noxyz_m.predict(Xf_noxyz_te), 1, 8)
m_noxyz     = evaluate(y_te, p_noxyz)
rf_all_r2   = results.get('RF-B (All CS)', {}).get('metrics', {'R2': float('nan')}).get('R2', float('nan'))
print(f'  RGB+HSV+LAB (no XYZ): R²={m_noxyz["R2"]:.4f}  RMSE={m_noxyz["RMSE"]:.3f}  rho={m_noxyz["Spearman_rho"]:.4f}')
print(f'  All CS (incl. XYZ):   R²={rf_all_r2:.4f}  (RF-B reference)')

xyz_delta = rf_all_r2 - m_noxyz['R2']
print(f'  XYZ incremental R2: {xyz_delta:+.4f}')
if abs(xyz_delta) < 0.01:
    print('  CONCLUSION: XYZ adds negligible predictive value over RGB+HSV+LAB.')
    print('     XYZ is a linear sRGB transform; Y~=L* (already in LAB); X and Z are')
    print('     RGB weighted sums with no additional perceptual separation.')
    print('     Recommendation: LAB+HSV are sufficient; include XYZ in combined')
    print('     feature set only if the delta justifies the added dimensions.')
elif xyz_delta > 0.01:
    print(f'  CONCLUSION: XYZ contributes +{xyz_delta:.4f} R2 over RGB+HSV+LAB.')
    print('     Investigate which XYZ features appear in top importances (Step 11).')
else:
    print(f'  CONCLUSION: Including XYZ slightly hurts performance ({xyz_delta:.4f} R2).')
    print('     XYZ introduces collinear dimensions that dilute feature selection.')


## 10 · Evaluation

### Primary regression metrics
For the composite colour-score prediction task (continuous regression → rounded to 1–8):

| Metric | Symbol | Interpretation |
|---|---|---|
| **R²** | Coefficient of determination | Variance explained by the model (0–1; higher is better) |
| **RMSE** | Root mean squared error | Average error in score units (lower is better) |
| **MAE** | Mean absolute error | Median-robust error in score units (lower is better) |
| **Spearman ρ** | Rank correlation | Whether relative ordering is preserved (0–1) |

> **REQ 6 — Accuracy reporting policy:**  
> `Accuracy` (exact score match after rounding) is **not reported for Composite Score prediction**.
> The composite score is a continuous regression target; accuracy on the discretized output
> is misleading (a prediction of 3.4 for a true score of 3 is accurate but rounds to 3 by luck;
> a prediction of 3.6 rounds to 4 and appears wrong). R² and RMSE are the correct metrics.
>
> **REQ 7 — Where Accuracy is appropriate:**  
> Accuracy (Acc@0 and Acc@1) is shown only in:
> - Confusion matrices (direct class-level visualisation)
> - The final pipeline summary table as a secondary diagnostic (Acc@1 = within-1-score accuracy)

All six pipelines (SVM-A, SVM-B, RF-A, RF-B, XGB-A, XGB-B) are compared side-by-side.


In [ ]:
# ── 10.1 Metrics summary table — R², RMSE, MAE, Spearman ρ (no raw Accuracy) ──
# REQ 6: Accuracy not reported for composite score prediction.
# REQ 7: Acc@1 (within-1-score) kept as a secondary diagnostic in the table.

rows = []
for name, res in results.items():
    m = res['metrics']
    rows.append({
        'Pipeline':      name,
        'Feature set':   res.get('feature_set', '—'),
        'Color space':   res.get('color_space', '—'),
        'R²':            round(m['R2'],           4),
        'RMSE':          round(m['RMSE'],          3),
        'MAE':           round(m['MAE'],           3),
        'Spearman ρ':    round(m['Spearman_rho'],  4),
        'Acc@1':         f"{m['Acc@1']:.1%}",
        'Train time (s)': round(res.get('train_time', float('nan')), 1),
    })

df_results = (pd.DataFrame(rows)
              .sort_values('R²', ascending=False)
              .reset_index(drop=True))
print('\n📊 Model Comparison — Primary metrics: R² and RMSE')
print('   (Acc@1 = within-1-score accuracy; shown as secondary diagnostic only)')
print(df_results.to_string(index=False))


In [ ]:
# ── 10.2 Scatter and residual plots (all 6 pipelines) ──────────────────────
# REQ 7: Accuracy shown here as part of the per-model annotation only, NOT
# as the primary evaluation criterion. Primary = R² and RMSE.
# Acc@1 (within ±1 score) annotated in the corner of each scatter plot.

#Confusion matrices (normalized and non-normalized)
n_models = len(results)
score_labels_cm = [str(i) for i in range(1, 9)]
cmap_cm = sns.color_palette('Blues', as_cmap=True)

# Non-normalized
ncols = min(4, n_models)
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
axes_flat = axes.ravel() if n_models > 1 else [axes]
for ax, (name, res) in zip(axes_flat, results.items()):
    cm = res['metrics']['CM']
    rho = res['metrics']['Spearman_rho']
    mae = res['metrics']['MAE']
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap_cm, ax=ax,
                xticklabels=score_labels_cm, yticklabels=score_labels_cm,
                linewidths=0.5, cbar=False)
    ax.set_title(f'{name}\nρ={rho:.3f}  MAE={mae:.2f}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for ax in axes_flat[n_models:]:
    ax.axis('off')
plt.suptitle('Confusion Matrices (Counts)', fontsize=14, fontweight='bold')
plt.tight_layout(); savefig('09a_confusion_matrices.png'); plt.show()

# Normalized
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
axes_flat = axes.ravel() if n_models > 1 else [axes]
for ax, (name, res) in zip(axes_flat, results.items()):
    cm = res['metrics']['CM']
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap=cmap_cm, ax=ax,
                xticklabels=score_labels_cm, yticklabels=score_labels_cm,
                linewidths=0.5, cbar=False, vmin=0, vmax=1)
    acc = res['metrics']['Accuracy']
    ax.set_title(f'{name}\nAcc={acc:.1%}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for ax in axes_flat[n_models:]:
    ax.axis('off')
plt.suptitle('Normalized Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout(); savefig('09b_confusion_matrices_normalized.png'); plt.show()

In [ ]:
# ── 10.3 Bar chart comparison — R², RMSE, MAE, Spearman ρ ──────────────────
# REQ 6: Accuracy bar not shown for composite-score regression comparison.

pipeline_names = list(results.keys())
maes  = [results[n]['metrics']['MAE']          for n in pipeline_names]
rmses = [results[n]['metrics']['RMSE']         for n in pipeline_names]
r2s   = [results[n]['metrics']['R2']           for n in pipeline_names]
rhos  = [results[n]['metrics']['Spearman_rho'] for n in pipeline_names]
acc1s = [results[n]['metrics']['Acc@1']        for n in pipeline_names]

# Colour by model family
palette = []
for n in pipeline_names:
    if   n.startswith('SVM'):  palette.append('#2980b9')
    elif n.startswith('RF'):   palette.append('#27ae60')
    else:                       palette.append('#e67e22')

x = np.arange(len(pipeline_names)); w = 0.55
fig, axes = plt.subplots(1, 4, figsize=(26, 5))

titles = ['R² (higher = better)', 'RMSE (lower = better)',
          'MAE (lower = better)', 'Spearman ρ (higher = better)']
vals_list = [r2s, rmses, maes, rhos]
for ax, title, vals in zip(axes, titles, vals_list):
    bars = ax.bar(x, vals, w, color=palette, edgecolor='white')
    for xi, v in enumerate(vals):
        ax.text(xi, v + 0.005, f'{v:.3f}', ha='center', fontsize=7, rotation=45)
    ax.set_xticks(x)
    ax.set_xticklabels(pipeline_names, rotation=40, ha='right', fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

# Legend for model families
import matplotlib.patches as mpatches
fig.legend(handles=[
    mpatches.Patch(color='#2980b9', label='SVM'),
    mpatches.Patch(color='#27ae60', label='Random Forest'),
    mpatches.Patch(color='#e67e22', label='XGBoost'),
], loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.08), fontsize=9)
plt.suptitle('Pipeline Comparison — R², RMSE, MAE, Spearman ρ', fontsize=13, fontweight='bold', y=1.10)
plt.tight_layout(); savefig('10_bar_comparison.png'); plt.show()

# Acc@1 shown separately (diagnostic, not primary metric)
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.bar(x, acc1s, w, color=palette, edgecolor='white')
for xi, v in enumerate(acc1s):
    ax2.text(xi, v + 0.005, f'{v:.1%}', ha='center', fontsize=8)
ax2.set_xticks(x); ax2.set_xticklabels(pipeline_names, rotation=40, ha='right', fontsize=9)
ax2.set_ylabel('Acc@1 (within ±1 score)')
ax2.set_title('Acc@1 — Within-1-Score Accuracy (secondary diagnostic)', fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout(); savefig('10b_acc1_secondary.png'); plt.show()
print('  ℹ️  Acc@1 shown as secondary diagnostic only — primary metrics are R² and RMSE.')


## 11 · Feature Analysis

Correlation heatmaps, feature importance for RF and XGBoost, and identification of the most important color-space features.

In [ ]:
# ── 11.1 Correlation heatmap (combined features, top-50 by variance) ─────────
df_comb = pd.DataFrame(X_feats_combined, columns=feat_names_combined)
# Select top 50 features by variance to keep heatmap readable
variances = df_comb.var().sort_values(ascending=False)
top_50 = variances.head(50).index.tolist()
corr_top = df_comb[top_50].corr()

fig, ax = plt.subplots(figsize=(16, 14))
tri = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, mask=tri, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            ax=ax, linewidths=0.2, annot=False,
            xticklabels=top_50, yticklabels=top_50)
ax.set_title('Feature Correlation Heatmap (Top 50 by variance)', fontsize=13, fontweight='bold')
ax.tick_params(labelsize=5)
plt.tight_layout(); savefig('11_correlation_heatmap_combined.png'); plt.show()

In [ ]:
# ── 11.2 Feature importance with channel and color-space breakdown ───────────
# REQ 4: Use importance/correlation evidence BEFORE removing any channel.
# REQ 3: XYZ importance rank explicitly shown.
#
# Train RF and XGBoost on all combined features; extract importances;
# break down by channel and color space; conclude which channels are
# dispensable (importance < threshold or consistently low across both models).

Xf_comb_tr_imp, pipes_comb_imp = build_feature_pipeline(
    X_feats_tr_aug_combined, y_tr_aug, k_best=80, scale=False)
Xf_comb_te_imp = apply_feature_pipeline(X_feats_te_combined, pipes_comb_imp)

# Recover selected feature names
vt_imp, cmask_imp, kb_imp, sc_imp = pipes_comb_imp
feat_after_vt_imp   = np.array(feat_names_combined)[vt_imp.get_support()]
feat_after_corr_imp = feat_after_vt_imp[cmask_imp]
feat_after_kb_imp   = feat_after_corr_imp[kb_imp.get_support()]

# RF importance model (global — reused in Step 12)
rf_imp_model_global = RandomForestRegressor(
    n_estimators=300, max_depth=12, random_state=SEED, n_jobs=-1)
rf_imp_model_global.fit(Xf_comb_tr_imp, y_tr_aug)
rf_importances_global = rf_imp_model_global.feature_importances_

# XGBoost importance model
xgb_imp_model_global = xgb.XGBRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    reg_lambda=10.0, reg_alpha=1.0, min_child_weight=5, gamma=1.0,
    subsample=0.8, colsample_bytree=0.7,
    random_state=SEED, n_jobs=-1, verbosity=0)
xgb_imp_model_global.fit(Xf_comb_tr_imp, y_tr_aug)
xgb_importances_global = xgb_imp_model_global.feature_importances_

# ── Breakdown by Color Space and Channel ─────────────────────────────────
n_show = min(30, len(feat_after_kb_imp))
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, imps, title, bar_color in [
    (axes[0], rf_importances_global,  'Random Forest', '#27ae60'),
    (axes[1], xgb_importances_global, 'XGBoost',       '#e67e22'),
]:
    top_idx = np.argsort(imps)[::-1][:n_show]
    feat_labels = [feat_after_kb_imp[i] for i in top_idx]
    # Color bars by color space
    cs_colors = {'RGB': '#e74c3c', 'HSV': '#8e44ad', 'LAB': '#2980b9', 'XYZ': '#e67e22'}
    bar_cols = [cs_colors.get(f.split('_')[0], '#7f8c8d') for f in feat_labels]
    ax.barh(range(n_show), imps[top_idx][::-1], color=bar_cols[::-1], edgecolor='white')
    ax.set_yticks(range(n_show))
    ax.set_yticklabels(feat_labels[::-1], fontsize=7)
    ax.set_xlabel('Importance'); ax.set_title(f'{title} — Top {n_show} Features', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

import matplotlib.patches as mpatches
fig.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='RGB'),
    mpatches.Patch(color='#8e44ad', label='HSV'),
    mpatches.Patch(color='#2980b9', label='LAB'),
    mpatches.Patch(color='#e67e22', label='XYZ'),
], loc='upper right', fontsize=9, ncol=2)
plt.suptitle('Feature Importance by Color Space and Channel', fontsize=13, fontweight='bold')
plt.tight_layout(); savefig('11b_feature_importance_by_cs.png'); plt.show()

# ── Per-CS and Per-channel importance summary ─────────────────────────────
print('\n📊 Importance breakdown by Color Space (sum of importance for top-80 features)')
print(f'{"CS":<8} {"RF importance":>15} {"XGB importance":>16} {"CS rank (RF)":>15}')
print('-' * 58)
cs_imp_rf  = {cs: 0.0 for cs in COLOR_SPACES}
cs_imp_xgb = {cs: 0.0 for cs in COLOR_SPACES}
for i, fname in enumerate(feat_after_kb_imp):
    cs_key = fname.split('_')[0]
    if cs_key in cs_imp_rf:
        cs_imp_rf[cs_key]  += float(rf_importances_global[i])
        cs_imp_xgb[cs_key] += float(xgb_importances_global[i])
for rank, (cs, _) in enumerate(sorted(cs_imp_rf.items(), key=lambda x: -x[1])):
    print(f'  {cs:<6}  {cs_imp_rf[cs]:>15.4f}  {cs_imp_xgb[cs]:>16.4f}  {rank+1:>15}')

# ── Per-channel importance: REQ 4 — justify retention/removal decisions ───
print('\n📊 Importance by Channel (to justify channel retention/removal)')
print('   Any channel with sum < 0.01 across both models is a candidate for removal.')
print(f'{"Channel":<15} {"RF importance":>15} {"XGB importance":>16} {"Decision":>12}')
print('-' * 60)
ch_imp_rf  = {}
ch_imp_xgb = {}
for i, fname in enumerate(feat_after_kb_imp):
    parts = fname.split('_')
    ch_key = '_'.join(parts[:2])  # e.g. 'LAB_b*'
    ch_imp_rf[ch_key]  = ch_imp_rf.get(ch_key, 0.0)  + float(rf_importances_global[i])
    ch_imp_xgb[ch_key] = ch_imp_xgb.get(ch_key, 0.0) + float(xgb_importances_global[i])
removal_threshold = 0.005
for ch in sorted(ch_imp_rf.keys()):
    rf_v  = ch_imp_rf[ch]
    xgb_v = ch_imp_xgb[ch]
    if rf_v < removal_threshold and xgb_v < removal_threshold:
        decision = 'CANDIDATE'
    else:
        decision = 'KEEP'
    print(f'  {ch:<13}  {rf_v:>15.4f}  {xgb_v:>16.4f}  {decision:>12}')
print()
print('  REQ 4: Channels marked CANDIDATE can be removed in future experiments.')
print('  Channels not marked CANDIDATE are RETAINED — no prior evidence justifies removal.')
print('  LAB-b*, HSV-V, and other channels are kept unless evidence shows they are dispensable.')


## 12 · Dimensionality Reduction

Two dimensionality reduction approaches compared, using the **best 2 models** by R² from Step 9:

| Method | Description | Dimensionality |
|---|---|---|
| **Baseline** | Full combined pipeline (288 → var+corr → top-K) | ~60–80 features |
| **PCA** | 95% explained variance on scaled combined features | auto (~10–30) |
| **Feature Selection** | Top-K by RF importance (same K as PCA, fair comparison) | same as PCA |

Goals:
1. Verify whether the feature space is redundant (PCA ≈ baseline)
2. Test whether RF importance-guided selection outperforms PCA
3. Guide further pipeline simplification for deployment

> **Leakage control:** PCA fitted on training set only. Feature importance indices
> derived from `rf_imp_model_global` trained on the training pipeline output only.


In [ ]:
# ── 12 · Dimensionality Reduction: PCA vs Feature Selection — Best 2 Pipelines ──
# Best 2 = top-2 by R² from results dict (all 6 pipelines compared in Step 10).
# Reports R² and RMSE only — no Accuracy (REQ 6).

# ── 12.1 PCA (95% variance) on combined scaled features ──────────────────────
scaler_pca  = StandardScaler().fit(X_feats_tr_aug_combined)
X_tr_scaled = scaler_pca.transform(X_feats_tr_aug_combined)
X_te_scaled = scaler_pca.transform(X_feats_te_combined)

pca = PCA(n_components=0.95, random_state=SEED)
X_tr_pca = pca.fit_transform(X_tr_scaled)
X_te_pca = pca.transform(X_te_scaled)
print(f'PCA (95% var): {X_feats_tr_aug_combined.shape[1]} → {X_tr_pca.shape[1]} components  '
      f'({pca.explained_variance_ratio_.sum()*100:.2f}% explained)')

# ── 12.2 Feature Selection (top-K by RF importance, K = #PCA components) ─────
Xf_comb_tr = Xf_comb_tr_imp
Xf_comb_te = Xf_comb_te_imp
rf_importances = rf_importances_global

n_select  = X_tr_pca.shape[1]
top_k_idx = np.argsort(rf_importances)[::-1][:n_select]
X_tr_fsel = Xf_comb_tr[:, top_k_idx]
X_te_fsel = Xf_comb_te[:, top_k_idx]
print(f'Feature Selection: top-{n_select} by RF importance')
print(f'  Top 5 features: {[feat_after_kb_imp[i] for i in top_k_idx[:5]]}')

# ── 12.3 Select best 2 pipelines from results (by R²) ────────────────────────
ranked = sorted(results.items(), key=lambda x: x[1]['metrics']['R2'], reverse=True)
best_2_names = [name for name, _ in ranked[:2]]
print(f'\nBest 2 pipelines (by R²): {best_2_names}')

def get_model_instance(name):
    bp = {k: v for k, v in results[name]['best_params'].items() if k != 'k_best'}
    if 'SVM' in name:
        return SVR(**bp, max_iter=15000, cache_size=500)
    elif 'RF'  in name:
        return RandomForestRegressor(**bp, random_state=SEED, n_jobs=-1)
    elif 'XGB' in name:
        return xgb.XGBRegressor(**bp, random_state=SEED, n_jobs=-1, verbosity=0)
    return None

# ── 12.4 Train & Evaluate all three representations × 2 pipelines ────────────
reduction_results = {}
for model_name in best_2_names:
    make_model = lambda mn=model_name: get_model_instance(mn)
    if make_model() is None:
        continue
    print(f'\n  {model_name}')

    for repr_name, Xtr, Xte, needs_scale in [
        ('Baseline (full pipeline)', Xf_comb_tr,  Xf_comb_te,  False),
        ('PCA (95% var)',             X_tr_pca,   X_te_pca,    False),
        (f'FeatSel (top-{n_select})', X_tr_fsel,  X_te_fsel,   False),
    ]:
        m_inst = make_model()
        if 'SVM' in model_name and not needs_scale:
            # SVM needs scaled input — scale on-the-fly
            sc_tmp = StandardScaler().fit(Xtr)
            m_inst.fit(sc_tmp.transform(Xtr), y_tr_aug)
            p = np.clip(m_inst.predict(sc_tmp.transform(Xte)), 1, 8)
        else:
            m_inst.fit(Xtr, y_tr_aug)
            p = np.clip(m_inst.predict(Xte), 1, 8)
        met = evaluate(y_te, p)
        key = f'{model_name} — {repr_name}'
        reduction_results[key] = met
        print(f'    {repr_name:<28}  R²={met["R2"]:.4f}  RMSE={met["RMSE"]:.3f}  ρ={met["Spearman_rho"]:.4f}')

# ── 12.5 Summary table and bar chart ─────────────────────────────────────────
red_rows = []
for rname, rmet in reduction_results.items():
    red_rows.append({
        'Pipeline × Method': rname,
        'R²':         round(rmet['R2'],           4),
        'RMSE':       round(rmet['RMSE'],          3),
        'MAE':        round(rmet['MAE'],           3),
        'Spearman ρ': round(rmet['Spearman_rho'],  4),
    })
df_reduction = pd.DataFrame(red_rows).sort_values('R²', ascending=False).reset_index(drop=True)
print('\n📊 Dimensionality Reduction — R² and RMSE (no Accuracy per REQ 6)')
print(df_reduction.to_string(index=False))

# Bar chart: R² and RMSE only
labels_r  = df_reduction['Pipeline × Method'].tolist()
r2_vals   = df_reduction['R²'].tolist()
rmse_vals = df_reduction['RMSE'].tolist()
palette_r = ['#2980b9' if 'Baseline' in l else ('#16a085' if 'PCA' in l else '#e67e22')
             for l in labels_r]

x = np.arange(len(labels_r)); w = 0.55
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, vals, title, flip in [
    (axes[0], r2_vals,   'R² (higher = better)',    False),
    (axes[1], rmse_vals, 'RMSE (lower = better)',   False),
]:
    ax.bar(x, vals, w, color=palette_r, edgecolor='white')
    for i, v in enumerate(vals):
        ax.text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=7, rotation=45)
    ax.set_xticks(x); ax.set_xticklabels(labels_r, rotation=45, ha='right', fontsize=7)
    ax.set_title(title, fontweight='bold'); ax.grid(axis='y', alpha=0.3)

import matplotlib.patches as mpatches
fig.legend(handles=[
    mpatches.Patch(color='#2980b9', label='Baseline (full pipeline)'),
    mpatches.Patch(color='#16a085', label='PCA'),
    mpatches.Patch(color='#e67e22', label='Feature Selection'),
], loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.06), fontsize=9)
plt.suptitle('Dimensionality Reduction Comparison — R² and RMSE', fontsize=13, fontweight='bold', y=1.08)
plt.tight_layout(); savefig('12_dim_reduction.png'); plt.show()


## 13 · Prediction Frieze

For **each model** (SVM-A, SVM-B, RF-A, RF-B, XGB-A, XGB-B), display a prediction frieze organized by score class:
- **Rows:** one per score class (scores 1–8 present in test set)
- **Columns:** up to **15 images** per score (test set)
- **Border colour:** 🟢 exact match  🟠 ±1 error  🔴 error > 1
- **Title per image:** `T:<true_score> P:<predicted_score>`


In [ ]:
# ── Prediction Frieze: one row per score, max 15 images, ALL models ──────────
#
# Layout: rows = scores 1–8, columns = up to 15 test images per score
# Border colour: 🟢 exact match  🟠 within-1 error  🔴 error > 1
# Each column header shows True (T) and Predicted (P) labels.

def prediction_frieze(images, y_true, y_pred, model_name, max_per_row=15):
    """
    Display a prediction frieze for one model.
    One row per score class, up to max_per_row images per row.
    Green border = exact; Orange = ±1; Red = error > 1.
    """
    y_true   = np.array(y_true, dtype=int)
    y_pred_r = np.clip(np.round(y_pred), 1, 8).astype(int)

    scores_present = sorted(set(y_true))
    n_rows  = len(scores_present)   # up to 8
    n_cols  = max_per_row            # 15

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 1.55, n_rows * 2.1))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    for row_idx, score in enumerate(scores_present):
        idxs = [i for i in range(len(y_true)) if y_true[i] == score]
        idxs = idxs[:n_cols]          # at most 15 images

        for col_idx in range(n_cols):
            ax = axes[row_idx, col_idx]
            if col_idx < len(idxs):
                i   = idxs[col_idx]
                err = abs(y_pred_r[i] - y_true[i])
                border_col = 'limegreen' if err == 0 else ('orange' if err == 1 else 'red')

                ax.imshow(images[i])
                ax.set_title(f'T:{y_true[i]} P:{y_pred_r[i]}',
                             fontsize=6.5, color=border_col, fontweight='bold', pad=2)
                for sp in ax.spines.values():
                    sp.set_visible(True)
                    sp.set_edgecolor(border_col)
                    sp.set_linewidth(2.5)
            else:
                ax.axis('off')
            ax.set_xticks([]); ax.set_yticks([])

        # Row label on the left
        axes[row_idx, 0].set_ylabel(
            f'Score {score}', fontsize=9, fontweight='bold',
            rotation=0, labelpad=50, va='center')

    # Legend patches at top
    import matplotlib.patches as mpatches
    legend_handles = [
        mpatches.Patch(color='limegreen', label='Exact match'),
        mpatches.Patch(color='orange',    label='±1 error'),
        mpatches.Patch(color='red',       label='>1 error'),
    ]
    fig.legend(handles=legend_handles, loc='upper right', fontsize=9,
               ncol=3, bbox_to_anchor=(0.99, 0.99), framealpha=0.9)

    plt.suptitle(
        f'Prediction Frieze — {model_name}\n',
        fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    fname_safe = model_name.replace(' ', '_').replace('/','_').lower()
    savefig(f'13_pred_frieze_{fname_safe}.png')
    plt.show()


# Run for every model
X_te_images = X_rgb_filtered[idx_te]

for model_name, res in results.items():
    prediction_frieze(X_te_images, y_te, res['preds'], model_name, max_per_row=15)
    print(f'  ✔ Frieze saved for {model_name}')


## 14 · Final Comparison Tables

### 14.1 Color space ablation (XGBoost benchmark)
### 14.2 Filtering method comparison
### 14.3 Master pipeline comparison table (REQ 8)

> **REQ 8:** Final table includes Model, Color space, Feature set, R², RMSE, Training time.
> Accuracy is omitted from this table per REQ 6 (not appropriate for regression scoring).
> Acc@1 is included as a supplementary diagnostic.


In [ ]:
# ── 14.1 Color Space Comparison ──────────────────────────────────────────────
# Train XGBoost on each color space separately to compare.
# Uses the same anti-overfitting settings as the main XGBoost (max_depth=5, strong reg).

cs_comparison_rows = []
for cs in COLOR_SPACES:
    X_tr_cs = X_feats_tr_aug_per_cs[cs]
    X_te_cs = np.nan_to_num(X_feats_per_cs[cs][idx_te], nan=0.0)
    Xf_tr_cs, pipes_cs = build_feature_pipeline(X_tr_cs, y_tr_aug, k_best=30)
    Xf_te_cs = apply_feature_pipeline(X_te_cs, pipes_cs)
    m_cs = xgb.XGBRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.75, colsample_bytree=0.7,
        reg_lambda=10.0, reg_alpha=1.0,
        min_child_weight=5, gamma=1.0,
        random_state=SEED, n_jobs=-1, verbosity=0)
    m_cs.fit(Xf_tr_cs, y_tr_aug)
    p_cs = np.clip(m_cs.predict(Xf_te_cs), 1, 8)
    met = evaluate(y_te, p_cs)
    cs_comparison_rows.append({
        'Color Space':  cs,
        'Model':        'XGBoost (benchmark)',
        'R²':           round(met['R2'],          3),
        'RMSE':         round(met['RMSE'],         3),
        'MAE':          round(met['MAE'],          3),
        'Spearman ρ':   round(met['Spearman_rho'], 4),
    })

# Add "Combined" row — pick the best Raw entry from reduction_results
# Keys look like "<ModelName> — Raw (full pipeline)", so filter by "Raw"
raw_entries = {k: v for k, v in reduction_results.items() if 'Raw' in k}
if raw_entries:
    # Use the best Raw result (highest Spearman ρ)
    best_raw_key  = max(raw_entries, key=lambda k: raw_entries[k]['Spearman_rho'])
    best_raw_met  = raw_entries[best_raw_key]
    cs_comparison_rows.append({
        'Color Space': f'Combined — {best_raw_key}',
            'MAE':         round(best_raw_met['MAE'], 3),
        'RMSE':        round(best_raw_met['RMSE'], 3),
        'R²':          round(best_raw_met['R2'], 3),
        'Spearman ρ':  round(best_raw_met['Spearman_rho'], 4),
    })
else:
    print("  ℹ️  No Raw entry in reduction_results; Combined row skipped.")

df_cs_comparison = pd.DataFrame(cs_comparison_rows)
print('\n📊 Color Space Comparison (XGBoost benchmark + best Raw combined)')
print(df_cs_comparison.sort_values('Spearman ρ', ascending=False).to_string(index=False))


In [ ]:
# ── 14.2 Filtering Method Comparison ─────────────────────────────────────────
filter_comparison_rows = []
for fname, fdata in X_feats_per_filter.items():
    X_tr_f = np.nan_to_num(fdata['train'], nan=0.0)
    X_te_f = np.nan_to_num(fdata['test'], nan=0.0)

    # Need to create augmented version for fair comparison
    # Use simple non-augmented train for filtering comparison
    Xf_tr_f, pipes_f = build_feature_pipeline(
        np.nan_to_num(np.vstack([X_tr_f] * 2), nan=0.0),  # double to avoid too few samples
        np.concatenate([y_tr_orig, y_tr_orig]),
        k_best=30)
    Xf_te_f = apply_feature_pipeline(X_te_f, pipes_f)

    m_f = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                            random_state=SEED, n_jobs=-1, verbosity=0)
    m_f.fit(Xf_tr_f, np.concatenate([y_tr_orig, y_tr_orig]))
    p_f = np.clip(m_f.predict(Xf_te_f), 1, 8)
    met_f = evaluate(y_te, p_f)
    filter_comparison_rows.append({
        'Filtering': fname,
        'Accuracy': f"{met_f['Accuracy']:.1%}",
        'MAE': round(met_f['MAE'], 3),
        'RMSE': round(met_f['RMSE'], 3),
        'R²': round(met_f['R2'], 3),
        'Spearman ρ': round(met_f['Spearman_rho'], 4),
    })

df_filter_comparison = pd.DataFrame(filter_comparison_rows)
print('\n📊 Filtering Method Comparison (XGBoost, LAB features)')
print(df_filter_comparison.to_string(index=False))

In [ ]:
# ── 14.3 Master Pipeline Comparison Table (REQ 8) ───────────────────────────
# Columns: Model | Color space | Feature set | R² | RMSE | Training time
# Accuracy intentionally excluded per REQ 6 (not appropriate for regression scoring).

master_rows = []
for name, res in results.items():
    m = res['metrics']
    master_rows.append({
        'Pipeline':       name,
        'Color space':    res.get('color_space', '—'),
        'Feature set':    res.get('feature_set', '—'),
        'R²':             round(m['R2'],   4),
        'RMSE':           round(m['RMSE'], 3),
        'MAE':            round(m['MAE'],  3),
        'Spearman ρ':     round(m['Spearman_rho'], 4),
        'Acc@1 (diag.)':  f"{m['Acc@1']:.1%}",
        'Train time (s)': round(res.get('train_time', float('nan')), 1),
    })

df_master = (pd.DataFrame(master_rows)
             .sort_values('R²', ascending=False)
             .reset_index(drop=True))

print('\n' + '='*80)
print('MASTER PIPELINE COMPARISON TABLE (REQ 8)')
print('Primary metrics: R² and RMSE  |  Acc@1 shown as diagnostic only')
print('='*80)
print(df_master.to_string(index=False))

# Bar chart: R² and RMSE only (REQ 6)
labels = df_master['Pipeline'].tolist()
r2_vals   = df_master['R²'].tolist()
rmse_vals = df_master['RMSE'].tolist()
time_vals = df_master['Train time (s)'].tolist()
palette2 = ['#2980b9' if 'SVM' in l else ('#27ae60' if 'RF' in l else '#e67e22')
            for l in labels]
x = np.arange(len(labels)); w = 0.55

fig, axes = plt.subplots(1, 3, figsize=(22, 5))

axes[0].bar(x, r2_vals, w, color=palette2, edgecolor='white')
for i, v in enumerate(r2_vals):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=7, rotation=45)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
axes[0].set_ylabel('R²'); axes[0].set_title('R² by Pipeline (higher = better)', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x, rmse_vals, w, color=palette2, edgecolor='white')
for i, v in enumerate(rmse_vals):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=7, rotation=45)
axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
axes[1].set_ylabel('RMSE (score units)')
axes[1].set_title('RMSE by Pipeline (lower = better)', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(x, time_vals, w, color=palette2, edgecolor='white')
for i, v in enumerate(time_vals):
    if not np.isnan(v):
        axes[2].text(i, v + 0.5, f'{v:.0f}s', ha='center', fontsize=7, rotation=45)
axes[2].set_xticks(x); axes[2].set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
axes[2].set_ylabel('Training time (s)')
axes[2].set_title('Training Time by Pipeline', fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)

import matplotlib.patches as mpatches
fig.legend(handles=[
    mpatches.Patch(color='#2980b9', label='SVM'),
    mpatches.Patch(color='#27ae60', label='Random Forest'),
    mpatches.Patch(color='#e67e22', label='XGBoost'),
], loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.06), fontsize=9)
plt.suptitle('Master Pipeline Comparison — R², RMSE, Training Time', fontsize=13, fontweight='bold', y=1.08)
plt.tight_layout(); savefig('14_master_comparison.png'); plt.show()

# Export
master_csv = os.path.join(OUT_DIR, 'master_pipeline_comparison.csv')
df_master.to_csv(master_csv, index=False)
print(f'\n  Master table exported to {master_csv}')

# Also export all results to Excel (multi-sheet)
excel_path = os.path.join(OUT_DIR, 'all_results_v3.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_master.to_excel(writer, sheet_name='Master Comparison', index=False)
    df_cs_comparison.to_excel(writer, sheet_name='Color Spaces', index=False)
    df_filter_comparison.to_excel(writer, sheet_name='Filtering Methods', index=False)
    df_gauss.to_excel(writer, sheet_name='Gaussianity Tests', index=False)
print(f'  Excel export: {excel_path}')


## 15 · Final Analysis

In [ ]:
# ── 15.1 Final Markdown Analysis ────────────────────────────────────────────
# Programmatically generates a summary report from all results.
# Primary metrics: R² and RMSE (REQ 6).
# Accuracy intentionally omitted from composite-score evaluation.

best_row    = df_master.iloc[0]
best_name   = best_row['Pipeline']
best_r2     = best_row['R²']
best_rmse   = best_row['RMSE']

best_cs_row = (df_cs_comparison.sort_values('R²', ascending=False).iloc[0]
               if 'df_cs_comparison' in dir() else None)
best_cs     = best_cs_row['Color Space'] if best_cs_row is not None else '—'
best_cs_r2  = best_cs_row['R²']          if best_cs_row is not None else '—'

best_filt_row = (df_filter_comparison.sort_values('R²', ascending=False).iloc[0]
                 if 'df_filter_comparison' in dir() else None)
best_filter   = best_filt_row['Filtering'] if best_filt_row is not None else '—'
best_filt_r2  = best_filt_row['R²']        if best_filt_row is not None else '—'

# Build analysis markdown
analysis_md = '# Zucchini Color Scoring — Final Analysis Report\n\n'

analysis_md += '## 1. Best Overall Pipeline\n'
analysis_md += f'**{best_name}** achieved the highest R² = {best_r2} (RMSE = {best_rmse}).\n\n'
analysis_md += '| Pipeline | Color space | Feature set | R² | RMSE | Train time (s) |\n'
analysis_md += '|---|---|---|---|---|---|\n'
for _, row in df_master.iterrows():
    analysis_md += (f"| {row['Pipeline']} | {row['Color space']} | {row['Feature set']} "
                    f"| {row['R²']} | {row['RMSE']} | {row['Train time (s)']} |\n")

analysis_md += '\n## 2. Color Space Analysis\n'
analysis_md += f'Best individual color space (XGBoost benchmark): **{best_cs}** (R² = {best_cs_r2}).\n\n'
if best_cs_row is not None:
    analysis_md += '| Color Space | R² | RMSE | Spearman ρ |\n'
    analysis_md += '|---|---|---|---|\n'
    for _, row in df_cs_comparison.sort_values('R²', ascending=False).iterrows():
        analysis_md += f"| {row['Color Space']} | {row['R²']} | {row['RMSE']} | {row['Spearman ρ']} |\n"

analysis_md += '\n### XYZ Color Space\n'
analysis_md += ('XYZ was investigated via: (a) standalone model, (b) ablation (all CS minus XYZ vs all CS), '
                '(c) channel correlation with score, and (d) feature importance ranking (Step 11).\n'
                'XYZ is a linear transform of sRGB; Y ≈ CIE luminance (≈ LAB L*); X and Z are '
                'weighted sums of RGB channels. The incremental contribution of XYZ over LAB+HSV '
                'is quantified in Step 9.5.\n')

analysis_md += '\n## 3. Filtering Strategy\n'
analysis_md += f'Best filter: **{best_filter}** (R² = {best_filt_r2}).\n'
if best_filt_row is not None:
    analysis_md += '| Filter | R² | RMSE |\n|---|---|---|\n'
    for _, row in df_filter_comparison.sort_values('R²', ascending=False).iterrows():
        analysis_md += f"| {row['Filtering']} | {row['R²']} | {row['RMSE']} |\n"

analysis_md += '\n## 4. Model-Specific Pipeline Insights\n'
analysis_md += ('Each model was trained with a scientifically motivated feature pipeline:\n'
                '- **SVM-A** (LAB + HSV-S texture + StandardScaler): RBF kernel benefits from '
                    'perceptually uniform LAB distances; HSV-S adds saturation signal.\n'
                '- **SVM-B** (LAB only): baseline to quantify HSV-S contribution.\n'
                '- **RF-A** (HSV + LAB, no scaling): scale-invariant; avoids XYZ/RGB collinearity.\n'
                '- **RF-B** (All 4 CS): tests whether broader features help random forest.\n'
                '- **XGB-A** (All CS combined): broadest feature set with strong regularisation.\n'
                '- **XGB-B** (LAB only): LAB baseline for XGBoost.\n')

analysis_md += '\n## 5. Channel Retention Policy (REQ 4)\n'
analysis_md += ('All channels (including LAB-b*, HSV-V) were retained unless feature importance '
                'analysis (Step 11) showed sum importance < 0.005 across both RF and XGBoost. '
                'Channels marked CANDIDATE in Step 11 are documented for future removal experiments. '
                'No channel was removed a priori.\n')

analysis_md += '\n## 6. Evaluation Metric Policy\n'
analysis_md += ('Primary metrics: **R²** and **RMSE** (regression evaluation). '
                'Classification Accuracy is not reported for composite-score prediction — '
                'it is misleading for a continuous ordinal target. '
                'Acc@1 (within-±1-score) is shown as a secondary diagnostic only.\n')

# Display
from IPython.display import Markdown, display
display(Markdown(analysis_md))

# Save
report_path = os.path.join(OUT_DIR, 'final_analysis_report.md')
with open(report_path, 'w', encoding='utf-8') as _f:
    _f.write(analysis_md)
print(f'Analysis report saved to {report_path}')
